# Guide Excursions Monitor

Kaggle notebook for guide excursions fact-first monitoring and exporting `guide_excursions_results.json`.


In [ ]:
from __future__ import annotations

from pathlib import Path as _GuideNotebookPath
import sys as _GuideNotebookSys
_GUIDE_EMBEDDED_GOOGLE_AI = {"__init__.py": "\"\"\"Google AI SDK with rate limiting and secrets management.\n\nThis module provides:\n- SecretsProvider: Unified secret retrieval (env -> Kaggle Secrets -> encrypted datasets)\n- GoogleAIClient: Wrapper over google.generativeai with Supabase-based rate limiting\n- RateLimitError, ProviderError: Custom exceptions\n\"\"\"\n\nfrom google_ai.exceptions import RateLimitError, ProviderError\nfrom google_ai.secrets import SecretsProvider, get_secret, get_secret_pool\nfrom google_ai.client import GoogleAIClient\n\n__all__ = [\n    \"SecretsProvider\",\n    \"get_secret\",\n    \"get_secret_pool\",\n    \"GoogleAIClient\",\n    \"RateLimitError\",\n    \"ProviderError\",\n]\n", "client.py": "\"\"\"Google AI client with Supabase-based rate limiting.\n\nFeatures:\n- Wrapper over google.generativeai\n- Atomic reserve/finalize through Supabase RPC\n- NO_WAIT policy: raises RateLimitError immediately on limit exceeded\n- Retries only on provider errors (max 3)\n- Structured logging (JSON lines)\n- Idempotency via request_uid\n\"\"\"\n\nfrom __future__ import annotations\n\nimport asyncio\nimport json\nimport logging\nimport os\nimport random\nimport re\nimport time\nimport uuid\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone, timedelta\nfrom typing import Any, Callable, Optional, TYPE_CHECKING\nfrom time import monotonic as _monotonic\n\nfrom google_ai.exceptions import RateLimitError, ProviderError, ReservationError\n\nif TYPE_CHECKING:\n    from supabase import Client as SupabaseClient\n\nlogger = logging.getLogger(__name__)\nIncidentNotifier = Callable[[str, dict[str, Any]], Any]\n\n\n@dataclass\nclass ReserveResult:\n    \"\"\"Result of a successful rate limit reservation.\"\"\"\n    ok: bool\n    api_key_id: Optional[str] = None\n    env_var_name: Optional[str] = None\n    key_alias: Optional[str] = None\n    minute_bucket: Optional[str] = None\n    day_bucket: Optional[str] = None\n    limits: Optional[dict] = None\n    used_after: Optional[dict] = None\n    blocked_reason: Optional[str] = None\n    retry_after_ms: Optional[int] = None\n\n\n@dataclass\nclass UsageInfo:\n    \"\"\"Token usage information from provider response.\"\"\"\n    input_tokens: int = 0\n    output_tokens: int = 0\n    total_tokens: int = 0\n\n\n@dataclass\nclass RequestContext:\n    \"\"\"Context for a single request (may have multiple attempts).\"\"\"\n    request_uid: str\n    consumer: str\n    account_name: Optional[str]\n    model: str\n    reserved_tpm: int\n    requested_model: Optional[str] = None\n    provider_model: Optional[str] = None\n    provider_model_name: Optional[str] = None\n    api_key_id: Optional[str] = None\n    started_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))\n\n\nclass GoogleAIClient:\n    \"\"\"Google AI client with rate limiting and retry logic.\n    \n    Usage:\n        client = GoogleAIClient(\n            supabase_client=get_supabase_client(),\n            secrets_provider=get_provider(),\n        )\n        \n        response = await client.generate_content_async(\n            model=\"gemma-3-27b\",\n            prompt=\"Hello, world!\",\n        )\n    \"\"\"\n    \n    # Default values\n    DEFAULT_MAX_OUTPUT_TOKENS = 8192\n    DEFAULT_TPM_RESERVE_EXTRA = 1000\n    # Heuristic budget for prompt token estimation. We intentionally overestimate\n    # to avoid passing Supabase reserve checks and then hitting provider 429\n    # on input-token-per-minute quotas.\n    _BYTES_PER_TOKEN_ESTIMATE = 4.0\n    MAX_RETRIES = 3\n    RETRY_DELAYS_MS = [250, 500, 1000]  # Backoff delays\n    RESERVE_FALLBACK_ENV = \"GOOGLE_AI_ALLOW_RESERVE_FALLBACK\"\n    LOCAL_FALLBACK_ENV = \"GOOGLE_AI_LOCAL_LIMITER_FALLBACK\"\n    LOCAL_FALLBACK_ON_ERROR_ENV = \"GOOGLE_AI_LOCAL_LIMITER_ON_RESERVE_ERROR\"\n    LOCAL_RPM_ENV = \"GOOGLE_AI_LOCAL_RPM\"\n    LOCAL_TPM_ENV = \"GOOGLE_AI_LOCAL_TPM\"\n    LOCAL_RPD_ENV = \"GOOGLE_AI_LOCAL_RPD\"\n    RESERVE_RPC_RECHECK_ENV = \"GOOGLE_AI_RESERVE_RPC_RECHECK_SECONDS\"\n    RESERVE_RPC_RETRY_ATTEMPTS_ENV = \"GOOGLE_AI_RESERVE_RPC_RETRY_ATTEMPTS\"\n    RESERVE_RPC_RETRY_BASE_DELAY_MS_ENV = \"GOOGLE_AI_RESERVE_RPC_RETRY_BASE_DELAY_MS\"\n    INCIDENT_NOTIFICATIONS_ENV = \"GOOGLE_AI_INCIDENT_NOTIFICATIONS\"\n    INCIDENT_COOLDOWN_ENV = \"GOOGLE_AI_INCIDENT_COOLDOWN_SECONDS\"\n    TEXT_PRIMARY_MODEL = \"gemma-3-27b\"\n    TEXT_MIN_GEMMA_B = 12\n    # When Supabase client is configured with a non-public schema, PostgREST can\n    # return 404 for RPC that exists in public. In that case we can retry via\n    # direct REST call with explicit schema headers.\n    RESERVE_DIRECT_RETRY_ENV = \"GOOGLE_AI_RESERVE_DIRECT_RETRY\"\n    RESERVE_DIRECT_SCHEMA_ENV = \"GOOGLE_AI_RESERVE_DIRECT_SCHEMA\"\n\n    # Process-local limiter (used when Supabase reserve RPC is missing/flaky).\n    _local_limiter_lock = asyncio.Lock()\n    _local_limiter_minute_bucket: int | None = None\n    _local_limiter_used_rpm: int = 0\n    _local_limiter_used_tpm: int = 0\n    _local_limiter_day_bucket: str | None = None\n    _local_limiter_used_rpd: int = 0\n\n    @staticmethod\n    def _normalize_rate_limit_model(model: str) -> str:\n        \"\"\"Normalize model id used in Supabase RPC quota tables.\n\n        Provider may use interactive Gemma variants (`-it`), while quota tables\n        in some projects store base model ids (`gemma-3-27b`).\n        \"\"\"\n        raw = (model or \"\").strip()\n        if raw.startswith(\"models/\"):\n            raw = raw.split(\"/\", 1)[1].strip()\n        if raw.startswith(\"gemma-\") and raw.endswith(\"-it\"):\n            return raw[:-3]\n        return raw\n\n    @classmethod\n    def _gemma_b_size(cls, model: str) -> Optional[int]:\n        normalized = cls._normalize_rate_limit_model(model).strip().lower()\n        match = re.match(r\"^gemma-\\d+(?:\\.\\d+)?-(\\d+)b$\", normalized)\n        if not match:\n            return None\n        try:\n            return int(match.group(1))\n        except Exception:\n            return None\n\n    @classmethod\n    def _is_gemma_model(cls, model: str) -> bool:\n        normalized = cls._normalize_rate_limit_model(model).strip().lower()\n        return normalized.startswith(\"gemma-\")\n\n    @classmethod\n    def _is_disallowed_text_model(cls, model: str) -> bool:\n        size = cls._gemma_b_size(model)\n        return size is not None and size < cls.TEXT_MIN_GEMMA_B\n\n    @staticmethod\n    def _resolve_provider_model(model: str) -> tuple[str, str]:\n        \"\"\"Resolve model id passed to provider and fully-qualified model_name.\n\n        Returns:\n            Tuple of (provider_model, provider_model_name):\n            - provider_model: short provider id (e.g. \"gemma-3-27b-it\")\n            - provider_model_name: fully-qualified API id (e.g. \"models/gemma-3-27b-it\")\n        \"\"\"\n        raw = (model or \"\").strip()\n        if raw.startswith(\"models/\"):\n            provider_model = raw.split(\"/\", 1)[1].strip() or raw\n            return provider_model, raw\n\n        provider_model = raw\n        if provider_model.startswith(\"gemma-\") and not provider_model.endswith(\"-it\"):\n            provider_model = f\"{provider_model}-it\"\n        return provider_model, f\"models/{provider_model}\"\n    \n    def __init__(\n        self,\n        supabase_client: Optional[\"SupabaseClient\"] = None,\n        secrets_provider: Optional[Any] = None,\n        consumer: str = \"bot\",\n        account_name: Optional[str] = None,\n        dry_run: bool = False,\n        incident_notifier: Optional[IncidentNotifier] = None,\n    ):\n        \"\"\"Initialize the client.\n        \n        Args:\n            supabase_client: Supabase client for rate limiting RPC calls\n            secrets_provider: Provider for API keys (if None, uses env directly)\n            consumer: Consumer identifier (bot/kaggle/script)\n            account_name: Account name for logging (from GOOGLE_API_LOCALNAME)\n            dry_run: If True, skip actual API calls (for testing)\n        \"\"\"\n        self.supabase = supabase_client\n        self.secrets_provider = secrets_provider\n        self.consumer = consumer\n        self.account_name = account_name or os.getenv(\"GOOGLE_API_LOCALNAME\")\n        self.dry_run = dry_run\n        self.allow_reserve_fallback = (\n            os.getenv(self.RESERVE_FALLBACK_ENV, \"1\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.allow_local_limiter_fallback = (\n            os.getenv(self.LOCAL_FALLBACK_ENV, \"1\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.allow_local_limiter_on_reserve_error = (\n            os.getenv(self.LOCAL_FALLBACK_ON_ERROR_ENV, \"1\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.incident_notifier = incident_notifier\n        self.incident_notifications_enabled = (\n            os.getenv(self.INCIDENT_NOTIFICATIONS_ENV, \"1\").strip().lower()\n            in {\"1\", \"true\", \"yes\", \"on\"}\n        )\n        self.incident_cooldown_seconds = self._read_int_env(\n            self.INCIDENT_COOLDOWN_ENV,\n            900,\n        )\n        self.reserve_rpc_recheck_seconds = self._read_int_env(\n            self.RESERVE_RPC_RECHECK_ENV,\n            60,\n        )\n        self.max_retries = self._read_int_env(\"GOOGLE_AI_MAX_RETRIES\", self.MAX_RETRIES)\n        self.retry_delays_ms = self._read_retry_delays()\n        self.fallback_models = self._read_fallback_models()\n        self._incident_last_sent: dict[str, float] = {}\n\n        # Cache missing Supabase RPCs to avoid noisy per-request fallbacks when\n        # the Supabase project hasn't been migrated yet (PGRST202).\n        self._reserve_rpc_missing = False\n        self._reserve_rpc_missing_since = 0.0\n        self._mark_sent_rpc_missing = False\n        self._finalize_rpc_missing = False\n        self._legacy_finalize_rpc_missing = False\n        self._missing_rpc_logged: set[str] = set()\n        \n        # Lazy import google.generativeai\n        self._genai = None\n\n    @staticmethod\n    def _read_int_env(name: str, default: int) -> int:\n        raw = (os.getenv(name) or \"\").strip()\n        if not raw:\n            return default\n        try:\n            value = int(raw)\n            return max(1, value)\n        except Exception:\n            return default\n\n    def _read_retry_delays(self) -> list[int]:\n        raw = (os.getenv(\"GOOGLE_AI_RETRY_DELAYS_MS\") or \"\").strip()\n        if not raw:\n            return list(self.RETRY_DELAYS_MS)\n        out: list[int] = []\n        for part in raw.split(\",\"):\n            p = part.strip()\n            if not p:\n                continue\n            try:\n                out.append(max(50, int(p)))\n            except Exception:\n                continue\n        return out or list(self.RETRY_DELAYS_MS)\n\n    async def _call_supabase_rpc_with_retries(\n        self,\n        fn_name: str,\n        payload: dict[str, Any],\n        *,\n        log_label: str,\n    ) -> Any:\n        \"\"\"Call a Supabase RPC with short retries on transient transport errors.\"\"\"\n        retry_attempts = self._read_int_env(self.RESERVE_RPC_RETRY_ATTEMPTS_ENV, 2)\n        retry_attempts = max(1, min(retry_attempts, 6))\n        retry_base_delay_ms = self._read_int_env(\n            self.RESERVE_RPC_RETRY_BASE_DELAY_MS_ENV,\n            350,\n        )\n        retry_base_delay_ms = max(50, min(retry_base_delay_ms, 5000))\n        last_exc: Exception | None = None\n        for rpc_attempt in range(1, retry_attempts + 1):\n            try:\n                return self.supabase.rpc(fn_name, payload).execute()\n            except Exception as exc:\n                last_exc = exc\n                transient = self._is_transient_reserve_rpc_error(exc)\n                if not transient or rpc_attempt >= retry_attempts:\n                    raise\n                delay_ms = int(retry_base_delay_ms * (2 ** (rpc_attempt - 1)))\n                delay_ms += random.randint(0, max(30, retry_base_delay_ms // 2))\n                delay_ms = min(delay_ms, 7000)\n                logger.warning(\n                    \"google_ai.%s_rpc_transient_retry fn=%s attempt=%s/%s delay_ms=%s err=%s\",\n                    log_label,\n                    fn_name,\n                    rpc_attempt,\n                    retry_attempts,\n                    delay_ms,\n                    exc,\n                )\n                await asyncio.sleep(delay_ms / 1000.0)\n        raise last_exc or RuntimeError(f\"RPC failed: {fn_name}\")\n\n    @staticmethod\n    def _is_transient_reserve_rpc_error(exc: Exception) -> bool:\n        cls_name = exc.__class__.__name__.lower()\n        msg = str(exc or \"\").lower()\n        if \"timeout\" in cls_name or \"ssl\" in cls_name or \"connection\" in cls_name:\n            return True\n        markers = (\n            \"timed out\",\n            \"timeout\",\n            \"handshake\",\n            \"server disconnected\",\n            \"connection reset\",\n            \"connection aborted\",\n            \"eof\",\n            \"unexpected eof\",\n            \"temporarily unavailable\",\n            \"tls\",\n            \"ssl\",\n        )\n        return any(token in msg for token in markers)\n\n    def _read_fallback_models(self) -> list[str]:\n        raw = (os.getenv(\"GOOGLE_AI_FALLBACK_MODELS\") or \"\").strip()\n        if not raw:\n            return []\n        out: list[str] = []\n        seen: set[str] = set()\n        for part in raw.split(\",\"):\n            model = part.strip()\n            if not model:\n                continue\n            key = model.lower()\n            if key in seen:\n                continue\n            seen.add(key)\n            out.append(model)\n        return out\n\n    def _build_model_chain(self, requested_model: str) -> list[str]:\n        requested = (requested_model or \"\").strip()\n        chain: list[str] = []\n        seen: set[str] = set()\n        for model in [requested, *self.fallback_models]:\n            m = (model or \"\").strip()\n            if not m:\n                continue\n            if self._is_disallowed_text_model(m):\n                logger.warning(\n                    \"google_ai.model_chain_skip model=%s reason=below_%sb_text_policy\",\n                    m,\n                    self.TEXT_MIN_GEMMA_B,\n                )\n                continue\n            key = self._normalize_rate_limit_model(m).lower()\n            if key in seen:\n                continue\n            seen.add(key)\n            chain.append(m)\n        has_gemma = self._is_gemma_model(requested) or any(\n            self._is_gemma_model(m) for m in self.fallback_models\n        )\n        if has_gemma:\n            primary = self.TEXT_PRIMARY_MODEL\n            primary_key = self._normalize_rate_limit_model(primary).lower()\n            head = next(\n                (m for m in chain if self._normalize_rate_limit_model(m).lower() == primary_key),\n                primary,\n            )\n            tail = [\n                m\n                for m in chain\n                if self._normalize_rate_limit_model(m).lower() != primary_key\n            ]\n            chain = [head, *tail]\n        return chain or [self.TEXT_PRIMARY_MODEL if has_gemma else requested_model]\n\n    async def _notify_incident(\n        self,\n        kind: str,\n        *,\n        ctx: RequestContext | None = None,\n        severity: str = \"critical\",\n        message: str | None = None,\n        details: dict[str, Any] | None = None,\n    ) -> None:\n        if not self.incident_notifier or not self.incident_notifications_enabled:\n            return\n\n        model = (ctx.requested_model if ctx else None) or (ctx.model if ctx else None) or \"\"\n        base = f\"{kind}:{self.consumer}:{model}\"\n        dedupe_key = base\n        if details:\n            code = details.get(\"error_code\") or details.get(\"blocked_reason\") or details.get(\"error_type\")\n            if code:\n                dedupe_key = f\"{base}:{code}\"\n\n        now = _monotonic()\n        last = self._incident_last_sent.get(dedupe_key)\n        if last is not None and (now - last) < float(self.incident_cooldown_seconds):\n            return\n        self._incident_last_sent[dedupe_key] = now\n\n        payload: dict[str, Any] = {\n            \"kind\": kind,\n            \"severity\": severity,\n            \"consumer\": self.consumer,\n            \"account_name\": self.account_name,\n            \"message\": message,\n            \"ts\": datetime.now(timezone.utc).isoformat(),\n        }\n        if ctx:\n            payload.update(\n                {\n                    \"request_uid\": ctx.request_uid,\n                    \"model\": ctx.model,\n                    \"requested_model\": ctx.requested_model or ctx.model,\n                    \"provider_model\": ctx.provider_model,\n                    \"provider_model_name\": ctx.provider_model_name,\n                    \"invoked_model\": ctx.provider_model_name or ctx.requested_model or ctx.model,\n                }\n            )\n        if details:\n            payload.update(details)\n        try:\n            maybe = self.incident_notifier(kind, payload)\n            if asyncio.iscoroutine(maybe):\n                await maybe\n        except Exception as exc:\n            logger.warning(\"google_ai incident notifier failed: %s\", exc)\n    \n    @property\n    def genai(self):\n        \"\"\"Lazy-load google.generativeai module.\"\"\"\n        if self._genai is None:\n            try:\n                import google.generativeai as genai\n                self._genai = genai\n            except ImportError:\n                raise ImportError(\n                    \"google-generativeai package not installed. \"\n                    \"Install with: pip install google-generativeai\"\n                )\n        return self._genai\n    \n    async def generate_content_async(\n        self,\n        model: str,\n        prompt: str,\n        generation_config: Optional[dict] = None,\n        safety_settings: Optional[list] = None,\n        max_output_tokens: Optional[int] = None,\n        candidate_key_ids: Optional[list[str]] = None,\n    ) -> tuple[str, UsageInfo]:\n        \"\"\"Generate content with rate limiting and retries.\n        \n        Args:\n            model: Model name (e.g., \"gemma-3-27b\")\n            prompt: Input prompt\n            generation_config: Optional generation config\n            safety_settings: Optional safety settings\n            max_output_tokens: Max output tokens (for TPM reservation)\n            candidate_key_ids: Optional list of API key IDs to try\n            \n        Returns:\n            Tuple of (response_text, usage_info)\n            \n        Raises:\n            RateLimitError: If rate limits exceeded (NO_WAIT)\n            ProviderError: If provider error after max retries\n        \"\"\"\n        request_uid = str(uuid.uuid4())\n        reserved_tpm = self._calculate_reserved_tpm(\n            prompt=prompt,\n            max_output_tokens=max_output_tokens or self.DEFAULT_MAX_OUTPUT_TOKENS,\n        )\n        requested_model = (model or \"\").strip()\n        model_chain = self._build_model_chain(requested_model)\n        attempt_cursor = 0\n\n        last_error: Optional[Exception] = None\n        for model_index, model_name in enumerate(model_chain):\n            limit_model = self._normalize_rate_limit_model(model_name)\n            provider_model, provider_model_name = self._resolve_provider_model(\n                model_name or limit_model\n            )\n            ctx = RequestContext(\n                request_uid=request_uid,\n                consumer=self.consumer,\n                account_name=self.account_name,\n                model=limit_model,\n                requested_model=model_name or limit_model,\n                provider_model=provider_model,\n                provider_model_name=provider_model_name,\n                reserved_tpm=reserved_tpm,\n            )\n\n            for local_attempt_no in range(1, self.max_retries + 1):\n                attempt_cursor += 1\n                attempt_no = attempt_cursor\n                try:\n                    return await self._attempt_generate(\n                        ctx=ctx,\n                        attempt_no=attempt_no,\n                        prompt=prompt,\n                        generation_config=generation_config,\n                        safety_settings=safety_settings,\n                        max_output_tokens=max_output_tokens,\n                        candidate_key_ids=candidate_key_ids,\n                    )\n                except RateLimitError as e:\n                    if (e.blocked_reason or \"\").strip().lower() in {\"no_keys\", \"model_not_found\"}:\n                        await self._notify_incident(\n                            \"rate_limit_blocked\",\n                            ctx=ctx,\n                            severity=\"critical\",\n                            message=str(e),\n                            details={\n                                \"blocked_reason\": e.blocked_reason,\n                                \"retry_after_ms\": e.retry_after_ms,\n                            },\n                        )\n                    raise\n                except ReservationError as e:\n                    last_error = e\n                    await self._notify_incident(\n                        \"reservation_error\",\n                        ctx=ctx,\n                        severity=\"critical\",\n                        message=str(e),\n                    )\n                    raise\n                except ProviderError as e:\n                    last_error = e\n                    # Keep provider-side 429 fail-fast here. Higher-level flows\n                    # already decide whether to wait, defer, or retry, and an\n                    # extra retry loop in the client multiplies end-to-end delay.\n                    if int(getattr(e, \"status_code\", 0) or 0) == 429:\n                        raise\n                    can_retry = bool(e.retryable) and local_attempt_no < self.max_retries\n                    if can_retry:\n                        delay_ms = self.retry_delays_ms[\n                            min(local_attempt_no - 1, len(self.retry_delays_ms) - 1)\n                        ]\n                        if e.retry_after_ms:\n                            delay_ms = max(int(delay_ms), int(e.retry_after_ms))\n                        jitter_ms = random.randint(0, 100)\n                        await asyncio.sleep((delay_ms + jitter_ms) / 1000)\n                        self._log_event(\"google_ai.retry\", ctx, attempt_no=attempt_no, error=str(e))\n                        continue\n\n                    has_fallback = model_index < (len(model_chain) - 1)\n                    await self._notify_incident(\n                        \"provider_error_fallback\" if has_fallback else \"provider_error\",\n                        ctx=ctx,\n                        severity=\"warning\" if has_fallback else \"critical\",\n                        message=str(e),\n                        details={\n                            \"error_type\": e.error_type,\n                            \"error_code\": e.error_code,\n                            \"status_code\": e.status_code,\n                            \"retryable\": int(bool(e.retryable)),\n                            \"attempt_no\": attempt_no,\n                            \"max_retries\": self.max_retries,\n                            \"next_model\": model_chain[model_index + 1] if has_fallback else None,\n                        },\n                    )\n                    if has_fallback:\n                        self._log_event(\n                            \"google_ai.model_fallback\",\n                            ctx,\n                            attempt_no=attempt_no,\n                            next_model=model_chain[model_index + 1],\n                            error=e,\n                        )\n                        break\n                    raise\n\n        raise last_error or ProviderError(error_type=\"unknown\", error_message=\"Max retries exceeded\")\n    \n    async def _attempt_generate(\n        self,\n        ctx: RequestContext,\n        attempt_no: int,\n        prompt: str,\n        generation_config: Optional[dict],\n        safety_settings: Optional[list],\n        max_output_tokens: Optional[int],\n        candidate_key_ids: Optional[list[str]],\n    ) -> tuple[str, UsageInfo]:\n        \"\"\"Single attempt to generate content.\"\"\"\n        \n        # 1. Reserve rate limit slot\n        reserve_result = await self._reserve(ctx, attempt_no, candidate_key_ids)\n        \n        if not reserve_result.ok:\n            raise RateLimitError(\n                blocked_reason=reserve_result.blocked_reason or \"unknown\",\n                retry_after_ms=reserve_result.retry_after_ms,\n                model=ctx.model,\n                api_key_id=reserve_result.api_key_id,\n                minute_bucket=reserve_result.minute_bucket,\n                day_bucket=reserve_result.day_bucket,\n            )\n\n        ctx.api_key_id = reserve_result.api_key_id\n        self._log_event(\"google_ai.reserve_ok\", ctx, attempt_no=attempt_no, reserve=reserve_result)\n        \n        # 2. Get API key\n        api_key = self._get_api_key(reserve_result.env_var_name)\n        if not api_key:\n            await self._notify_incident(\n                \"missing_api_key\",\n                ctx=ctx,\n                severity=\"critical\",\n                message=f\"API key not found: {reserve_result.env_var_name}\",\n                details={\"env_var_name\": reserve_result.env_var_name},\n            )\n            raise ReservationError(f\"API key not found: {reserve_result.env_var_name}\")\n        \n        # 3. Mark as sent (before actual call)\n        await self._mark_sent(ctx, attempt_no)\n        \n        # 4. Call provider\n        start_time = _monotonic()\n        try:\n            if self.dry_run:\n                # Dry run mode for testing\n                response_text = f\"[DRY RUN] Response for: {prompt[:50]}...\"\n                usage = UsageInfo(input_tokens=100, output_tokens=50, total_tokens=150)\n            else:\n                response_text, usage = await self._call_provider(\n                    api_key=api_key,\n                    model=ctx.requested_model or ctx.model,\n                    prompt=prompt,\n                    generation_config=generation_config,\n                    safety_settings=safety_settings,\n                    max_output_tokens=max_output_tokens,\n                )\n            \n            duration_ms = int((_monotonic() - start_time) * 1000)\n            \n        except Exception as e:\n            duration_ms = int((_monotonic() - start_time) * 1000)\n            \n            # Classify error\n            provider_error = self._classify_error(e)\n            \n            # Finalize with error\n            await self._finalize(\n                ctx=ctx,\n                attempt_no=attempt_no,\n                usage=None,\n                duration_ms=duration_ms,\n                error=provider_error,\n            )\n            \n            self._log_event(\n                \"google_ai.call_error\",\n                ctx,\n                attempt_no=attempt_no,\n                duration_ms=duration_ms,\n                error=provider_error,\n            )\n            \n            raise provider_error\n        \n        # 5. Finalize (update usage, reconcile TPM)\n        await self._finalize(\n            ctx=ctx,\n            attempt_no=attempt_no,\n            usage=usage,\n            duration_ms=duration_ms,\n        )\n        \n        self._log_event(\n            \"google_ai.call_ok\",\n            ctx,\n            attempt_no=attempt_no,\n            duration_ms=duration_ms,\n            usage=usage,\n        )\n        \n        return response_text, usage\n    \n    async def _reserve(\n        self,\n        ctx: RequestContext,\n        attempt_no: int,\n        candidate_key_ids: Optional[list[str]],\n    ) -> ReserveResult:\n        \"\"\"Reserve rate limit slot via Supabase RPC.\"\"\"\n        if not self.supabase:\n            # No Supabase = no rate limiting (for local dev)\n            logger.warning(\"No Supabase client, skipping rate limit reservation\")\n            return ReserveResult(\n                ok=True,\n                env_var_name=\"GOOGLE_API_KEY\",\n            )\n        was_cached_missing = self._reserve_rpc_missing\n        if was_cached_missing:\n            now = _monotonic()\n            age = (\n                now - self._reserve_rpc_missing_since\n                if self._reserve_rpc_missing_since > 0\n                else 0.0\n            )\n            if age < self.reserve_rpc_recheck_seconds:\n                if self.allow_local_limiter_fallback:\n                    return await self._local_reserve(\n                        ctx,\n                        attempt_no=attempt_no,\n                        key_alias=\"local-fallback-no-rpc-cached\",\n                        blocked_reason=\"reserve_rpc_missing\",\n                    )\n                return ReserveResult(\n                    ok=True,\n                    env_var_name=\"GOOGLE_API_KEY\",\n                    key_alias=\"reserve-fallback-no-rpc-cached\",\n                    blocked_reason=\"reserve_rpc_missing\",\n                )\n            # Cooldown elapsed: retry RPC once instead of staying in cached fallback forever.\n            self._reserve_rpc_missing = False\n            logger.warning(\n                \"google_ai.reserve_rpc_recheck consumer=%s model=%s age_s=%.1f\",\n                ctx.consumer,\n                ctx.model,\n                age,\n            )\n\n        if self._reserve_rpc_missing:\n            if self.allow_local_limiter_fallback:\n                return await self._local_reserve(\n                    ctx,\n                    attempt_no=attempt_no,\n                    key_alias=\"local-fallback-no-rpc-cached\",\n                    blocked_reason=\"reserve_rpc_missing\",\n                )\n            return ReserveResult(\n                ok=True,\n                env_var_name=\"GOOGLE_API_KEY\",\n                key_alias=\"reserve-fallback-no-rpc-cached\",\n                blocked_reason=\"reserve_rpc_missing\",\n            )\n        \n        payload = {\n            \"p_request_uid\": ctx.request_uid,\n            \"p_attempt_no\": attempt_no,\n            \"p_consumer\": ctx.consumer,\n            \"p_account_name\": ctx.account_name,\n            \"p_model\": ctx.model,\n            \"p_reserved_tpm\": ctx.reserved_tpm,\n            \"p_candidate_key_ids\": candidate_key_ids,\n        }\n\n        try:\n            retry_attempts = self._read_int_env(self.RESERVE_RPC_RETRY_ATTEMPTS_ENV, 2)\n            retry_attempts = max(1, min(retry_attempts, 6))\n            retry_base_delay_ms = self._read_int_env(self.RESERVE_RPC_RETRY_BASE_DELAY_MS_ENV, 350)\n            retry_base_delay_ms = max(50, min(retry_base_delay_ms, 5000))\n            rpc_error: Exception | None = None\n            result = None\n            for rpc_attempt in range(1, retry_attempts + 1):\n                try:\n                    result = self.supabase.rpc(\"google_ai_reserve\", payload).execute()\n                    rpc_error = None\n                    break\n                except Exception as exc:\n                    rpc_error = exc\n                    transient = self._is_transient_reserve_rpc_error(exc)\n                    if not transient or rpc_attempt >= retry_attempts:\n                        raise\n                    delay_ms = int(retry_base_delay_ms * (2 ** (rpc_attempt - 1)))\n                    delay_ms += random.randint(0, max(30, retry_base_delay_ms // 2))\n                    delay_ms = min(delay_ms, 7000)\n                    logger.warning(\n                        \"google_ai.reserve_rpc_transient_retry consumer=%s model=%s attempt=%s/%s delay_ms=%s err=%s\",\n                        ctx.consumer,\n                        ctx.model,\n                        rpc_attempt,\n                        retry_attempts,\n                        delay_ms,\n                        str(exc)[:260],\n                    )\n                    await asyncio.sleep(delay_ms / 1000.0)\n            if result is None:\n                if rpc_error:\n                    raise rpc_error\n                raise RuntimeError(\"google_ai_reserve returned no result\")\n            \n            data = result.data\n            if isinstance(data, list) and data:\n                data = data[0]\n\n            if was_cached_missing:\n                logger.info(\n                    \"google_ai.reserve_rpc_recovered consumer=%s model=%s\",\n                    ctx.consumer,\n                    ctx.model,\n                )\n                self._reserve_rpc_missing_since = 0.0\n                self._missing_rpc_logged.discard(\"google_ai_reserve\")\n            \n            return ReserveResult(\n                ok=data.get(\"ok\", False),\n                api_key_id=data.get(\"api_key_id\"),\n                env_var_name=data.get(\"env_var_name\"),\n                key_alias=data.get(\"key_alias\"),\n                minute_bucket=data.get(\"minute_bucket\"),\n                day_bucket=data.get(\"day_bucket\"),\n                limits=data.get(\"limits\"),\n                used_after=data.get(\"used_after\"),\n                blocked_reason=data.get(\"blocked_reason\"),\n                retry_after_ms=data.get(\"retry_after_ms\"),\n            )\n            \n        except Exception as e:\n            if (\n                self.allow_reserve_fallback\n                and self._is_missing_reserve_rpc_error(e)\n                and (not self.dry_run)\n                and (os.getenv(self.RESERVE_DIRECT_RETRY_ENV, \"1\").strip().lower() in {\"1\", \"true\", \"yes\", \"on\"})\n            ):\n                # Retry via direct REST call with explicit schema headers.\n                direct = await self._reserve_via_direct_rest(ctx, attempt_no=attempt_no, payload=payload)\n                if direct is not None:\n                    if was_cached_missing:\n                        self._reserve_rpc_missing_since = 0.0\n                        self._missing_rpc_logged.discard(\"google_ai_reserve\")\n                    return direct\n\n            if self.allow_reserve_fallback and self._is_missing_reserve_rpc_error(e):\n                msg = str(e)\n                self._reserve_rpc_missing = True\n                self._reserve_rpc_missing_since = _monotonic()\n                if \"google_ai_reserve\" not in self._missing_rpc_logged:\n                    self._missing_rpc_logged.add(\"google_ai_reserve\")\n                    logger.error(\n                        \"Supabase RPC google_ai_reserve is missing in this Supabase project \"\n                        \"(PGRST202). Rate limiting via Supabase is disabled; using direct env key \"\n                        \"GOOGLE_API_KEY instead. Set %s=0 to fail hard. error=%s\",\n                        self.RESERVE_FALLBACK_ENV,\n                        msg,\n                    )\n                self._log_event(\n                    \"google_ai.reserve_fallback_no_rpc\",\n                    ctx,\n                    attempt_no=attempt_no,\n                    error=msg[:500],\n                )\n                await self._notify_incident(\n                    \"reserve_rpc_missing\",\n                    ctx=ctx,\n                    severity=\"warning\",\n                    message=(\n                        \"Supabase RPC google_ai_reserve missing; \"\n                        \"switched to process-local limiter fallback (direct API key).\"\n                    ),\n                    details={\"error\": msg[:500]},\n                )\n                if self.allow_local_limiter_fallback:\n                    return await self._local_reserve(\n                        ctx,\n                        attempt_no=attempt_no,\n                        key_alias=\"local-fallback\",\n                        blocked_reason=\"reserve_rpc_missing\",\n                    )\n                return ReserveResult(\n                    ok=True,\n                    env_var_name=\"GOOGLE_API_KEY\",\n                    key_alias=\"reserve-fallback\",\n                    blocked_reason=\"reserve_rpc_missing\",\n                )\n            if (\n                self.allow_reserve_fallback\n                and self.allow_local_limiter_on_reserve_error\n                and self.allow_local_limiter_fallback\n            ):\n                msg = str(e)\n                logger.warning(\"Reserve RPC failed; using local limiter fallback. error=%s\", msg)\n                await self._notify_incident(\n                    \"reserve_rpc_error_fallback\",\n                    ctx=ctx,\n                    severity=\"warning\",\n                    message=f\"Reserve RPC failed; using local limiter fallback: {e}\",\n                    details={\"error\": msg[:500]},\n                )\n                return await self._local_reserve(\n                    ctx,\n                    attempt_no=attempt_no,\n                    key_alias=\"local-fallback-reserve-error\",\n                    blocked_reason=\"reserve_rpc_error\",\n                )\n            logger.error(\"Failed to call google_ai_reserve: %s\", e)\n            await self._notify_incident(\n                \"reserve_rpc_error\",\n                ctx=ctx,\n                severity=\"critical\",\n                message=f\"Reserve RPC failed: {e}\",\n                details={\"error\": str(e)[:500]},\n            )\n            raise ReservationError(f\"Reserve RPC failed: {e}\")\n\n    async def _reserve_via_direct_rest(\n        self,\n        ctx: RequestContext,\n        *,\n        attempt_no: int,\n        payload: dict[str, Any],\n    ) -> ReserveResult | None:\n        \"\"\"Call google_ai_reserve via REST endpoint, forcing schema headers.\n\n        This is a fallback for environments where Supabase client is configured with\n        a different schema (e.g. 'private') and PostgREST returns 404 for an RPC\n        that exists in 'public'.\n        \"\"\"\n        base_url = (os.getenv(\"SUPABASE_URL\") or \"\").strip().rstrip(\"/\")\n        key = (os.getenv(\"SUPABASE_SERVICE_KEY\") or os.getenv(\"SUPABASE_KEY\") or \"\").strip()\n        if not base_url or not key:\n            return None\n        schema = (os.getenv(self.RESERVE_DIRECT_SCHEMA_ENV) or \"public\").strip() or \"public\"\n        endpoint = f\"{base_url}/rest/v1/rpc/google_ai_reserve\"\n        headers = {\n            \"apikey\": key,\n            \"Authorization\": f\"Bearer {key}\",\n            \"Content-Type\": \"application/json\",\n            \"Accept\": \"application/json\",\n            \"Accept-Profile\": schema,\n            \"Content-Profile\": schema,\n        }\n\n        def _do() -> tuple[int, str]:\n            import requests\n\n            resp = requests.post(endpoint, headers=headers, json=payload, timeout=20)\n            return int(resp.status_code), resp.text or \"\"\n\n        try:\n            status, body = await asyncio.to_thread(_do)\n        except Exception as exc:\n            self._log_event(\n                \"google_ai.reserve_direct_error\",\n                ctx,\n                attempt_no=attempt_no,\n                error=str(exc)[:300],\n            )\n            return None\n\n        if status == 404:\n            # Still missing in this schema/project.\n            return None\n        if status >= 400:\n            self._log_event(\n                \"google_ai.reserve_direct_http_error\",\n                ctx,\n                attempt_no=attempt_no,\n                status=status,\n                body_head=(body or \"\").replace(\"\\n\", \" \")[:240],\n            )\n            return None\n\n        try:\n            data = json.loads(body) if body else {}\n            if isinstance(data, list) and data:\n                data = data[0]\n            if not isinstance(data, dict):\n                return None\n        except Exception:\n            return None\n\n        self._log_event(\n            \"google_ai.reserve_direct_ok\",\n            ctx,\n            attempt_no=attempt_no,\n            status=status,\n            schema=schema,\n        )\n        return ReserveResult(\n            ok=bool(data.get(\"ok\", False)),\n            api_key_id=data.get(\"api_key_id\"),\n            env_var_name=data.get(\"env_var_name\"),\n            key_alias=data.get(\"key_alias\") or f\"direct:{schema}\",\n            minute_bucket=data.get(\"minute_bucket\"),\n            day_bucket=data.get(\"day_bucket\"),\n            limits=data.get(\"limits\"),\n            used_after=data.get(\"used_after\"),\n            blocked_reason=data.get(\"blocked_reason\"),\n            retry_after_ms=data.get(\"retry_after_ms\"),\n        )\n\n    def _read_local_limit(self, name: str, default: int) -> int:\n        raw = (os.getenv(name) or \"\").strip()\n        if not raw:\n            return default\n        try:\n            return max(1, int(float(raw)))\n        except Exception:\n            return default\n\n    async def _local_reserve(\n        self,\n        ctx: RequestContext,\n        *,\n        attempt_no: int,\n        key_alias: str,\n        blocked_reason: str,\n    ) -> ReserveResult:\n        \"\"\"Process-local limiter used when Supabase reserve RPC is missing/flaky.\"\"\"\n        rpm_limit = self._read_local_limit(self.LOCAL_RPM_ENV, 20)\n        tpm_limit = self._read_local_limit(self.LOCAL_TPM_ENV, 12000)\n        rpd_limit = self._read_local_limit(self.LOCAL_RPD_ENV, 5000)\n\n        now = time.time()\n        minute_bucket = int(now // 60) * 60\n        day_bucket = datetime.fromtimestamp(now, timezone.utc).date().isoformat()\n        required_tpm = max(1, int(ctx.reserved_tpm))\n\n        async with self._local_limiter_lock:\n            if self._local_limiter_minute_bucket != minute_bucket:\n                self._local_limiter_minute_bucket = minute_bucket\n                self._local_limiter_used_rpm = 0\n                self._local_limiter_used_tpm = 0\n            if self._local_limiter_day_bucket != day_bucket:\n                self._local_limiter_day_bucket = day_bucket\n                self._local_limiter_used_rpd = 0\n\n            limits = {\"rpd\": rpd_limit, \"rpm\": rpm_limit, \"tpm\": tpm_limit}\n            used_before = {\n                \"rpd\": self._local_limiter_used_rpd,\n                \"rpm\": self._local_limiter_used_rpm,\n                \"tpm\": self._local_limiter_used_tpm,\n            }\n\n            if self._local_limiter_used_rpd + 1 > rpd_limit:\n                midnight = (\n                    datetime.fromtimestamp(now, timezone.utc)\n                    .replace(hour=0, minute=0, second=0, microsecond=0)\n                    + timedelta(days=1)\n                )\n                retry_after_ms = max(1000, int((midnight.timestamp() - now) * 1000))\n                return ReserveResult(\n                    ok=False,\n                    env_var_name=\"GOOGLE_API_KEY\",\n                    key_alias=key_alias,\n                    minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                    day_bucket=day_bucket,\n                    limits=limits,\n                    used_after=used_before,\n                    blocked_reason=\"rpd\",\n                    retry_after_ms=retry_after_ms,\n                )\n            if self._local_limiter_used_rpm + 1 > rpm_limit:\n                retry_after_ms = max(250, int((minute_bucket + 60 - now) * 1000))\n                return ReserveResult(\n                    ok=False,\n                    env_var_name=\"GOOGLE_API_KEY\",\n                    key_alias=key_alias,\n                    minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                    day_bucket=day_bucket,\n                    limits=limits,\n                    used_after=used_before,\n                    blocked_reason=\"rpm\",\n                    retry_after_ms=retry_after_ms,\n                )\n            if self._local_limiter_used_tpm + required_tpm > tpm_limit:\n                retry_after_ms = max(250, int((minute_bucket + 60 - now) * 1000))\n                return ReserveResult(\n                    ok=False,\n                    env_var_name=\"GOOGLE_API_KEY\",\n                    key_alias=key_alias,\n                    minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                    day_bucket=day_bucket,\n                    limits=limits,\n                    used_after=used_before,\n                    blocked_reason=\"tpm\",\n                    retry_after_ms=retry_after_ms,\n                )\n\n            self._local_limiter_used_rpd += 1\n            self._local_limiter_used_rpm += 1\n            self._local_limiter_used_tpm += required_tpm\n            used_after = {\n                \"rpd\": self._local_limiter_used_rpd,\n                \"rpm\": self._local_limiter_used_rpm,\n                \"tpm\": self._local_limiter_used_tpm,\n            }\n\n            reserve = ReserveResult(\n                ok=True,\n                env_var_name=\"GOOGLE_API_KEY\",\n                key_alias=key_alias,\n                minute_bucket=datetime.fromtimestamp(minute_bucket, timezone.utc).isoformat(),\n                day_bucket=day_bucket,\n                limits=limits,\n                used_after=used_after,\n                blocked_reason=blocked_reason,\n                retry_after_ms=None,\n            )\n\n        self._log_event(\"google_ai.reserve_local_fallback_ok\", ctx, attempt_no=attempt_no, reserve=reserve)\n        return reserve\n\n    @staticmethod\n    def _is_missing_reserve_rpc_error(error: Exception) -> bool:\n        return GoogleAIClient._is_missing_rpc_error(error, \"google_ai_reserve\")\n\n    @staticmethod\n    def _is_missing_rpc_error(error: Exception, rpc_name: str) -> bool:\n        message = str(error).lower()\n        rpc_name_l = rpc_name.lower()\n        if rpc_name_l not in message:\n            return False\n        markers = (\n            \"pgrst202\",\n            \"route post:/rpc/\",\n            \"not found\",\n            \"schema cache\",\n        )\n        return any(marker in message for marker in markers)\n    \n    async def _mark_sent(self, ctx: RequestContext, attempt_no: int) -> None:\n        \"\"\"Mark request as sent (before calling provider).\"\"\"\n        if not self.supabase:\n            return\n        if self._mark_sent_rpc_missing:\n            return\n        request_uid = ctx.request_uid\n\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_mark_sent\",\n                {\n                    \"p_request_uid\": request_uid,\n                    \"p_attempt_no\": attempt_no,\n                },\n                log_label=\"mark_sent\",\n            )\n        except Exception as e:\n            message = str(e)\n            if self._is_missing_rpc_error(e, \"google_ai_mark_sent\"):\n                self._mark_sent_rpc_missing = True\n                if \"google_ai_mark_sent\" not in self._missing_rpc_logged:\n                    self._missing_rpc_logged.add(\"google_ai_mark_sent\")\n                    logger.warning(\n                        \"Supabase RPC google_ai_mark_sent is missing (PGRST202). \"\n                        \"Will skip it for the rest of the process. error=%s\",\n                        message,\n                    )\n                return\n            logger.warning(\"Failed to mark_sent: %s\", e)\n            await self._notify_incident(\n                \"mark_sent_rpc_error\",\n                ctx=ctx,\n                severity=\"warning\",\n                message=message,\n                details={\"attempt_no\": attempt_no, \"rpc\": \"google_ai_mark_sent\"},\n            )\n    \n    async def _finalize(\n        self,\n        ctx: RequestContext,\n        attempt_no: int,\n        usage: Optional[UsageInfo],\n        duration_ms: int,\n        error: Optional[ProviderError] = None,\n    ) -> None:\n        \"\"\"Finalize request (record usage, reconcile TPM).\"\"\"\n        if not self.supabase:\n            return\n\n        legacy_payload = {\n            \"p_request_uid\": ctx.request_uid,\n            \"p_api_key_id\": ctx.api_key_id,\n            \"p_model\": ctx.model,\n            \"p_actual_input_tokens\": usage.input_tokens if usage else None,\n            \"p_actual_output_tokens\": usage.output_tokens if usage else None,\n            \"p_status\": \"success\" if not error else \"failed\",\n        }\n\n        if self._finalize_rpc_missing:\n            await self._finalize_legacy(legacy_payload)\n            return\n\n        payload = {\n            \"p_request_uid\": ctx.request_uid,\n            \"p_attempt_no\": attempt_no,\n            \"p_usage_input_tokens\": usage.input_tokens if usage else None,\n            \"p_usage_output_tokens\": usage.output_tokens if usage else None,\n            \"p_usage_total_tokens\": usage.total_tokens if usage else None,\n            \"p_duration_ms\": duration_ms,\n            \"p_provider_status\": \"succeeded\" if not error else \"failed\",\n            \"p_error_type\": error.error_type if error else None,\n            \"p_error_code\": error.error_code if error else None,\n            \"p_error_message\": error.error_message if error else None,\n        }\n\n        try:\n            await self._call_supabase_rpc_with_retries(\n                \"google_ai_finalize\",\n                payload,\n                log_label=\"finalize\",\n            )\n            return\n        except Exception as e:\n            if not self._is_missing_rpc_error(e, \"google_ai_finalize\"):\n                logger.warning(\"Failed to finalize: %s\", e)\n                await self._notify_incident(\n                    \"finalize_rpc_error\",\n                    ctx=ctx,\n                    severity=\"warning\",\n                    message=str(e),\n                    details={\"attempt_no\": attempt_no, \"rpc\": \"google_ai_finalize\"},\n                )\n                return\n            logger.info(\"google_ai_finalize missing, falling back to finalize_google_ai_usage\")\n            # Don't try google_ai_finalize again in this process.\n            self._finalize_rpc_missing = True\n\n        await self._finalize_legacy(legacy_payload)\n\n    async def _finalize_legacy(self, legacy_payload: dict[str, Any]) -> None:\n        \"\"\"Legacy finalize fallback for Supabase projects without google_ai_finalize.\"\"\"\n        if self._legacy_finalize_rpc_missing:\n            return\n\n        try:\n            self.supabase.rpc(\"finalize_google_ai_usage\", legacy_payload).execute()\n        except Exception as legacy_error:\n            if self._is_missing_rpc_error(legacy_error, \"finalize_google_ai_usage\"):\n                self._legacy_finalize_rpc_missing = True\n                if \"finalize_google_ai_usage\" not in self._missing_rpc_logged:\n                    self._missing_rpc_logged.add(\"finalize_google_ai_usage\")\n                    logger.warning(\n                        \"Supabase RPC finalize_google_ai_usage is missing (PGRST202). \"\n                        \"Finalize is disabled for the rest of the process. error=%s\",\n                        legacy_error,\n                    )\n                return\n            logger.warning(\"Failed to finalize_google_ai_usage: %s\", legacy_error)\n    \n    async def _call_provider(\n        self,\n        api_key: str,\n        model: str,\n        prompt: str,\n        generation_config: Optional[dict],\n        safety_settings: Optional[list],\n        max_output_tokens: Optional[int],\n    ) -> tuple[str, UsageInfo]:\n        \"\"\"Call Google AI provider.\"\"\"\n        # Configure API key\n        self.genai.configure(api_key=api_key)\n        \n        # Build generation config\n        config = dict(generation_config or {})\n        if max_output_tokens and \"max_output_tokens\" not in config:\n            config[\"max_output_tokens\"] = max_output_tokens\n        \n        # Create model:\n        # - google.generativeai expects model_name like \"models/gemma-3-27b-it\"\n        # - For Gemma, \"-it\" is the tested interactive-tuned variant in this project.\n        # - For Gemini, use the model name as-is (no \"-it\" suffix).\n        _provider_model, model_name = self._resolve_provider_model(model)\n\n        # Gemma models frequently reject JSON-mode knobs like `response_mime_type`.\n        # Callers still request JSON in the prompt and then parse it from text.\n        if self._is_gemma_model(model_name) or self._is_gemma_model(model):\n            stripped = []\n            for key in (\"response_mime_type\", \"response_schema\", \"response_schema_name\"):\n                if key in config:\n                    stripped.append(key)\n                    config.pop(key, None)\n            if stripped:\n                logger.info(\n                    \"google_ai: stripped_generation_config model=%s provider_model=%s stripped=%s\",\n                    model,\n                    model_name,\n                    \",\".join(stripped),\n                )\n        gen_model = self.genai.GenerativeModel(model_name)\n        \n        # Generate content\n        response = await gen_model.generate_content_async(\n            prompt,\n            generation_config=config,\n            safety_settings=safety_settings,\n        )\n        \n        def _get_usage(resp: Any) -> UsageInfo:\n            usage = UsageInfo()\n            meta = getattr(resp, \"usage_metadata\", None)\n            if not meta:\n                return usage\n            try:\n                if isinstance(meta, dict):\n                    usage.input_tokens = int(meta.get(\"prompt_token_count\") or 0)\n                    usage.output_tokens = int(meta.get(\"candidates_token_count\") or 0)\n                    usage.total_tokens = int(meta.get(\"total_token_count\") or 0)\n                else:\n                    usage.input_tokens = int(getattr(meta, \"prompt_token_count\", 0) or 0)\n                    usage.output_tokens = int(getattr(meta, \"candidates_token_count\", 0) or 0)\n                    usage.total_tokens = int(getattr(meta, \"total_token_count\", 0) or 0)\n            except Exception:\n                # Best-effort only; token accounting must not break requests.\n                pass\n            return usage\n\n        def _extract_text(resp: Any) -> str:\n            # Old `google.generativeai`: response.text\n            try:\n                text = getattr(resp, \"text\", None)\n                if isinstance(text, str) and text.strip():\n                    return text.strip()\n            except Exception:\n                pass\n\n            # Newer responses often store content in candidates[].content.parts[].text\n            parts: list[str] = []\n            cands = getattr(resp, \"candidates\", None)\n            if cands:\n                for cand in list(cands):\n                    content = getattr(cand, \"content\", None)\n                    if content is None and isinstance(cand, dict):\n                        content = cand.get(\"content\")\n                    if content is None:\n                        continue\n                    cand_parts = getattr(content, \"parts\", None)\n                    if cand_parts is None and isinstance(content, dict):\n                        cand_parts = content.get(\"parts\")\n                    if cand_parts:\n                        for part in list(cand_parts):\n                            t = getattr(part, \"text\", None)\n                            if t is None and isinstance(part, dict):\n                                t = part.get(\"text\")\n                            if isinstance(t, str) and t.strip():\n                                parts.append(t.strip())\n                    else:\n                        t = getattr(content, \"text\", None)\n                        if isinstance(t, str) and t.strip():\n                            parts.append(t.strip())\n            if parts:\n                return \"\\n\".join(parts).strip()\n\n            # Last resort: stringify.\n            try:\n                return str(resp).strip()\n            except Exception:\n                return \"\"\n\n        usage = _get_usage(response)\n        response_text = _extract_text(response)\n        if not response_text:\n            raise ProviderError(\n                error_type=\"empty_response\",\n                error_message=(\n                    \"Provider returned empty text \"\n                    f\"(requested_model={model}, provider_model_name={model_name})\"\n                ),\n                retryable=True,\n            )\n        return response_text, usage\n    \n    def _get_api_key(self, env_var_name: Optional[str]) -> Optional[str]:\n        \"\"\"Get API key from environment or secrets provider.\"\"\"\n        name = env_var_name or \"GOOGLE_API_KEY\"\n        \n        if self.secrets_provider:\n            return self.secrets_provider.get_secret(name)\n        \n        return os.getenv(name)\n    \n    def _estimate_prompt_tokens(self, prompt: str) -> int:\n        \"\"\"Best-effort token estimate for prompts.\n\n        We can't depend on provider-side countTokens here (it would also require\n        an API call). We use conservative byte/char heuristics because long\n        Cyrillic/OCR prompts can tokenize much denser than a simple bytes/4\n        estimate and otherwise slip past reserve() only to hit provider 429.\n        \"\"\"\n        if not prompt:\n            return 1\n        try:\n            size = len(prompt.encode(\"utf-8\", errors=\"ignore\"))\n        except Exception:\n            size = len(prompt)\n        chars = len(prompt)\n        non_ascii = sum(1 for ch in prompt if ord(ch) > 127)\n        non_ascii_ratio = (non_ascii / chars) if chars > 0 else 0.0\n\n        bytes_est = size / float(self._BYTES_PER_TOKEN_ESTIMATE)\n        if non_ascii_ratio >= 0.30:\n            chars_est = chars * 0.72\n            bytes_est = size / 2.6\n        else:\n            chars_est = chars * 0.30\n\n        est = int(max(bytes_est, chars_est))\n        # Add overhead for JSON, escaping, and tokenization variance.\n        est = int(est * 1.15) + 50\n        return max(1, est)\n\n    def _calculate_reserved_tpm(self, *, prompt: str, max_output_tokens: int) -> int:\n        \"\"\"Calculate tokens to reserve for TPM check.\n\n        Supabase reservation must cover BOTH prompt (input) and output tokens.\n        Under-reserving here can lead to provider 429 (ResourceExhausted) even\n        when Supabase reserve() returned ok=true.\n        \"\"\"\n        input_est = self._estimate_prompt_tokens(prompt)\n        output_budget = max(1, int(max_output_tokens))\n        return input_est + output_budget + int(self.DEFAULT_TPM_RESERVE_EXTRA)\n    \n    def _classify_error(self, error: Exception) -> ProviderError:\n        \"\"\"Classify exception into ProviderError.\"\"\"\n        if isinstance(error, ProviderError):\n            return error\n        error_str = str(error)\n        error_lower = error_str.lower()\n        error_type = type(error).__name__\n\n        retry_after_ms: Optional[int] = None\n        # Gemini/Gemma errors often include \"Please retry in <seconds>s.\"\n        m_retry = re.search(r\"retry in\\s+(\\d+(?:\\.\\d+)?)\\s*s\", error_lower)\n        if m_retry:\n            try:\n                retry_after_ms = int(float(m_retry.group(1)) * 1000)\n            except Exception:\n                retry_after_ms = None\n        \n        # Check for retryable errors\n        retryable = any(x in error_lower for x in [\n            \"timeout\",\n            \"connection\",\n            \"temporary\",\n            \"rate limit\",\n            \"503\",\n            \"502\",\n            \"504\",\n            \"resource_exhausted\",\n            \"unavailable\",\n            \"deadline exceeded\",\n            \"internal\",\n            \"try again\",\n            \"econnreset\",\n            \"connection reset\",\n            \"socket\",\n        ])\n        status_code: Optional[int] = None\n        for code in (\"429\", \"500\", \"502\", \"503\", \"504\"):\n            if code in error_lower:\n                try:\n                    status_code = int(code)\n                except Exception:\n                    status_code = None\n                break\n        # Some exceptions don't include \"resource_exhausted\" in the string, but\n        # the type name is still informative.\n        if not retryable and error_type.lower() in {\"resourceexhausted\", \"unavailable\"}:\n            retryable = True\n        if not retryable and status_code == 429:\n            retryable = True\n        \n        return ProviderError(\n            error_type=error_type,\n            error_message=error_str[:500],  # Limit message length\n            retryable=retryable,\n            status_code=status_code,\n            retry_after_ms=retry_after_ms,\n        )\n    \n    def _log_event(\n        self,\n        event: str,\n        ctx: RequestContext,\n        attempt_no: int = 1,\n        **kwargs,\n    ) -> None:\n        \"\"\"Log structured event (JSON lines format).\"\"\"\n        log_data = {\n            \"ts\": datetime.now(timezone.utc).isoformat(),\n            \"event\": event,\n            \"request_uid\": ctx.request_uid,\n            \"attempt_no\": attempt_no,\n            \"consumer\": ctx.consumer,\n            \"account_name\": ctx.account_name,\n            \"model\": ctx.model,\n            \"requested_model\": ctx.requested_model or ctx.model,\n            \"provider_model\": ctx.provider_model,\n            \"provider_model_name\": ctx.provider_model_name,\n            \"invoked_model\": ctx.provider_model_name or ctx.requested_model or ctx.model,\n            \"reserved_tpm\": ctx.reserved_tpm,\n        }\n        \n        # Add optional fields\n        for key, value in kwargs.items():\n            if value is not None:\n                if hasattr(value, \"__dict__\"):\n                    log_data[key] = value.__dict__\n                else:\n                    log_data[key] = value\n        \n        logger.info(json.dumps(log_data, ensure_ascii=False, default=str))\n", "exceptions.py": "\"\"\"Custom exceptions for Google AI SDK.\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional\n\n\n@dataclass\nclass RateLimitError(Exception):\n    \"\"\"Raised when rate limits are exceeded.\n    \n    NO_WAIT policy: this error is raised immediately without waiting.\n    \"\"\"\n    blocked_reason: str  # 'rpm' | 'tpm' | 'rpd'\n    retry_after_ms: Optional[int] = None\n    model: Optional[str] = None\n    api_key_id: Optional[str] = None\n    minute_bucket: Optional[str] = None\n    day_bucket: Optional[str] = None\n    \n    def __str__(self) -> str:\n        msg = f\"Rate limit exceeded: {self.blocked_reason}\"\n        if self.retry_after_ms:\n            msg += f\" (retry after {self.retry_after_ms}ms)\"\n        return msg\n\n\n@dataclass\nclass ProviderError(Exception):\n    \"\"\"Raised when Google AI provider returns an error.\n    \n    Retryable errors will be retried up to 3 times.\n    \"\"\"\n    error_type: str\n    error_code: Optional[str] = None\n    error_message: Optional[str] = None\n    retryable: bool = False\n    status_code: Optional[int] = None\n    retry_after_ms: Optional[int] = None\n    \n    def __str__(self) -> str:\n        msg = f\"Provider error: {self.error_type}\"\n        if self.error_code:\n            msg += f\" ({self.error_code})\"\n        if self.error_message:\n            msg += f\": {self.error_message}\"\n        if self.retry_after_ms:\n            msg += f\" (retry after {self.retry_after_ms}ms)\"\n        return msg\n\n\nclass SecretsError(Exception):\n    \"\"\"Raised when secrets cannot be retrieved or decrypted.\"\"\"\n    pass\n\n\nclass ReservationError(Exception):\n    \"\"\"Raised when rate limit reservation fails unexpectedly.\"\"\"\n    pass\n", "secrets.py": "\"\"\"Secrets provider with fallback chain.\n\nFallback order:\n1. Environment variables (os.getenv)\n2. Kaggle Secrets (kaggle_secrets.UserSecretsClient)\n3. Encrypted datasets (Fernet + 2 private Kaggle datasets)\n\nBased on existing implementation in kaggle/UniversalFestivalParser/src/secrets.py\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom google_ai.exceptions import SecretsError\n\nlogger = logging.getLogger(__name__)\n\n\nclass SecretsProvider:\n    \"\"\"Unified secrets provider with fallback chain.\n    \n    Supports:\n    - Single secrets: get_secret(\"GOOGLE_API_KEY\")\n    - Secret pools: get_secret_pool(\"GOOGLE_API_KEY\") -> [\"key1\", \"key2\", ...]\n    \n    For pools, looks for GOOGLE_API_KEY, GOOGLE_API_KEY_2, GOOGLE_API_KEY_3, etc.\n    \"\"\"\n    \n    def __init__(\n        self,\n        cipher_dataset_path: str = \"/kaggle/input/eve-secrets-cipher\",\n        key_dataset_path: str = \"/kaggle/input/eve-secrets-key\",\n        # Legacy paths for backward compatibility\n        legacy_cipher_path: str = \"/kaggle/input/gemma-cipher\",\n        legacy_key_path: str = \"/kaggle/input/gemma-key\",\n    ):\n        self.cipher_dataset_path = cipher_dataset_path\n        self.key_dataset_path = key_dataset_path\n        self.legacy_cipher_path = legacy_cipher_path\n        self.legacy_key_path = legacy_key_path\n        self._cache: dict[str, str] = {}\n        self._bundle_loaded = False\n    \n    def get_secret(self, name: str) -> Optional[str]:\n        \"\"\"Get a secret by name.\n        \n        Fallback order:\n        1. Environment variable\n        2. Kaggle Secrets\n        3. Encrypted bundle from datasets\n        \n        Returns None if not found.\n        \"\"\"\n        # Check cache first\n        if name in self._cache:\n            return self._cache[name]\n        \n        # 1. Try environment variable\n        value = os.getenv(name)\n        if value:\n            logger.debug(\"Secret %s found in environment\", name)\n            self._cache[name] = value\n            return value\n        \n        # 2. Try Kaggle Secrets\n        value = self._get_from_kaggle_secrets(name)\n        if value:\n            logger.debug(\"Secret %s found in Kaggle Secrets\", name)\n            self._cache[name] = value\n            return value\n        \n        # 3. Try encrypted datasets\n        value = self._get_from_encrypted_bundle(name)\n        if value:\n            logger.debug(\"Secret %s found in encrypted bundle\", name)\n            self._cache[name] = value\n            return value\n        \n        logger.warning(\"Secret %s not found in any source\", name)\n        return None\n    \n    def get_secret_pool(self, prefix: str) -> list[str]:\n        \"\"\"Get all secrets matching a prefix pattern.\n        \n        For prefix \"GOOGLE_API_KEY\", returns values of:\n        - GOOGLE_API_KEY\n        - GOOGLE_API_KEY_2\n        - GOOGLE_API_KEY_3\n        - etc.\n        \n        Returns list of values (not keys).\n        \"\"\"\n        values = []\n        \n        # First key (no suffix)\n        value = self.get_secret(prefix)\n        if value:\n            values.append(value)\n        \n        # Numbered keys\n        for i in range(2, 100):  # Support up to 99 keys\n            value = self.get_secret(f\"{prefix}_{i}\")\n            if value:\n                values.append(value)\n            else:\n                break  # Stop on first missing\n        \n        return values\n    \n    def _get_from_kaggle_secrets(self, name: str) -> Optional[str]:\n        \"\"\"Try to get secret from Kaggle Secrets API.\"\"\"\n        try:\n            from kaggle_secrets import UserSecretsClient\n            secrets = UserSecretsClient()\n            value = secrets.get_secret(name)\n            if value:\n                return value\n        except ImportError:\n            logger.debug(\"kaggle_secrets not available\")\n        except Exception as e:\n            logger.debug(\"Kaggle Secrets error for %s: %s\", name, e)\n        return None\n    \n    def _get_from_encrypted_bundle(self, name: str) -> Optional[str]:\n        \"\"\"Try to get secret from encrypted dataset bundle.\"\"\"\n        if not self._bundle_loaded:\n            self._load_bundle()\n        return self._cache.get(name)\n    \n    def _load_bundle(self) -> None:\n        \"\"\"Load and decrypt secrets bundle from datasets.\"\"\"\n        self._bundle_loaded = True\n        \n        # Try new bundle format first\n        bundle = self._try_load_bundle(\n            self.cipher_dataset_path,\n            self.key_dataset_path,\n            bundle_file=\"secrets.enc\",\n            key_file=\"fernet.keys\",\n        )\n        \n        if bundle:\n            self._cache.update(bundle)\n            return\n        \n        # Try legacy format (single key file)\n        legacy_key = self._try_load_legacy_key()\n        if legacy_key:\n            self._cache[\"GOOGLE_API_KEY\"] = legacy_key\n    \n    def _try_load_bundle(\n        self,\n        cipher_path: str,\n        key_path: str,\n        bundle_file: str,\n        key_file: str,\n    ) -> Optional[dict[str, str]]:\n        \"\"\"Try to load and decrypt a secrets bundle.\"\"\"\n        cipher_file = Path(cipher_path) / bundle_file\n        keys_file = Path(key_path) / key_file\n        \n        if not cipher_file.exists() or not keys_file.exists():\n            return None\n        \n        try:\n            from cryptography.fernet import Fernet, MultiFernet\n            \n            # Load key ring (multiple Fernet keys for rotation)\n            key_lines = keys_file.read_text().strip().split(\"\\n\")\n            fernets = [Fernet(k.strip().encode()) for k in key_lines if k.strip()]\n            \n            if not fernets:\n                logger.warning(\"No valid Fernet keys in %s\", keys_file)\n                return None\n            \n            multi_fernet = MultiFernet(fernets)\n            \n            # Decrypt bundle\n            encrypted = cipher_file.read_bytes()\n            decrypted = multi_fernet.decrypt(encrypted)\n            bundle = json.loads(decrypted.decode(\"utf-8\"))\n            \n            logger.info(\"Loaded %d secrets from encrypted bundle\", len(bundle))\n            return bundle\n            \n        except ImportError:\n            logger.warning(\"cryptography package not installed\")\n        except Exception as e:\n            logger.error(\"Failed to decrypt bundle: %s\", e)\n        \n        return None\n    \n    def _try_load_legacy_key(self) -> Optional[str]:\n        \"\"\"Try to load single key from legacy dataset format.\"\"\"\n        cipher_file = Path(self.legacy_cipher_path) / \"google_api_key.enc\"\n        key_file = Path(self.legacy_key_path) / \"fernet.key\"\n        \n        if not cipher_file.exists() or not key_file.exists():\n            return None\n        \n        try:\n            from cryptography.fernet import Fernet\n            \n            fernet_key = key_file.read_bytes().strip()\n            fernet = Fernet(fernet_key)\n            \n            encrypted = cipher_file.read_bytes()\n            decrypted = fernet.decrypt(encrypted).decode(\"utf-8\").strip()\n            \n            # Validate Google API key format\n            if not decrypted.startswith(\"AIza\"):\n                logger.warning(\"Decrypted key has unexpected format\")\n            \n            logger.info(\"Loaded API key from legacy encrypted datasets\")\n            return decrypted\n            \n        except ImportError:\n            logger.warning(\"cryptography package not installed\")\n        except Exception as e:\n            logger.error(\"Failed to decrypt legacy key: %s\", e)\n        \n        return None\n\n\n# Module-level singleton\n_provider: Optional[SecretsProvider] = None\n\n\ndef get_provider() -> SecretsProvider:\n    \"\"\"Get or create the global secrets provider.\"\"\"\n    global _provider\n    if _provider is None:\n        _provider = SecretsProvider()\n    return _provider\n\n\ndef get_secret(name: str) -> Optional[str]:\n    \"\"\"Get a secret by name (module-level convenience function).\"\"\"\n    return get_provider().get_secret(name)\n\n\ndef get_secret_pool(prefix: str) -> list[str]:\n    \"\"\"Get all secrets matching a prefix (module-level convenience function).\"\"\"\n    return get_provider().get_secret_pool(prefix)\n\n\ndef create_encrypted_bundle(\n    secrets: dict[str, str],\n    output_dir: str | Path,\n    cipher_filename: str = \"secrets.enc\",\n    key_filename: str = \"fernet.keys\",\n) -> tuple[Path, Path]:\n    \"\"\"Create encrypted bundle files for Kaggle datasets.\n    \n    This is a helper function for setting up the datasets.\n    Run this locally to generate the files to upload.\n    \n    Args:\n        secrets: Dict of secret_name -> secret_value\n        output_dir: Directory to save files\n        cipher_filename: Name of cipher file\n        key_filename: Name of key file\n        \n    Returns:\n        Tuple of (cipher_file_path, key_file_path)\n    \"\"\"\n    from cryptography.fernet import Fernet\n    \n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    \n    # Generate new Fernet key\n    fernet_key = Fernet.generate_key()\n    fernet = Fernet(fernet_key)\n    \n    # Encrypt bundle as JSON\n    bundle_json = json.dumps(secrets, ensure_ascii=False)\n    encrypted = fernet.encrypt(bundle_json.encode(\"utf-8\"))\n    \n    # Save files\n    cipher_path = output_dir / cipher_filename\n    key_path = output_dir / key_filename\n    \n    cipher_path.write_bytes(encrypted)\n    key_path.write_text(fernet_key.decode(\"utf-8\"))\n    \n    print(f\"Created {cipher_path} (upload to eve-secrets-cipher dataset)\")\n    print(f\"Created {key_path} (upload to eve-secrets-key dataset)\")\n    print(\"IMPORTANT: Keep these datasets PRIVATE!\")\n    \n    return cipher_path, key_path\n"}
_GUIDE_EMBEDDED_ROOT = (_GuideNotebookPath.cwd() / 'embedded_repo_bundle').resolve()
_GUIDE_EMBEDDED_PACKAGE = _GUIDE_EMBEDDED_ROOT / 'google_ai'
_GUIDE_EMBEDDED_PACKAGE.mkdir(parents=True, exist_ok=True)
for _guide_name, _guide_body in _GUIDE_EMBEDDED_GOOGLE_AI.items():
    (_GUIDE_EMBEDDED_PACKAGE / _guide_name).write_text(_guide_body, encoding='utf-8')
if str(_GUIDE_EMBEDDED_ROOT) not in _GuideNotebookSys.path:
    _GuideNotebookSys.path.insert(0, str(_GUIDE_EMBEDDED_ROOT))
__file__ = str((_GuideNotebookPath.cwd() / 'guide_excursions_monitor.py').resolve())

import asyncio
import base64
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_REPO = WORK_DIR / "repo_bundle"
RESULT_PATH = WORK_DIR / "guide_excursions_results.json"
REPO_BOOTSTRAP_WAIT_SECONDS = max(0, int(os.getenv("GUIDE_MONITORING_REPO_BOOTSTRAP_WAIT_SECONDS", "15") or 15))

SCRIPT_DIR = Path(__file__).resolve().parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))


def ensure_libs() -> None:
    modules = [
        ("telethon", "telethon"),
        ("google.generativeai", "google-generativeai"),
        ("cryptography", "cryptography"),
        ("supabase", "supabase"),
        ("nest_asyncio", "nest_asyncio"),
    ]
    missing: list[str] = []
    for module_name, package_name in modules:
        try:
            __import__(module_name)
        except Exception:
            missing.append(package_name)
    if missing:
        print(f"Installing Python packages: {', '.join(missing)}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Python packages already available: telethon, google-generativeai, cryptography, supabase", flush=True)


ensure_libs()


def _bootstrap_repo_bundle() -> None:
    bundled_package = SCRIPT_DIR / "google_ai" / "__init__.py"
    if bundled_package.exists():
        print(f"Using bundled google_ai from {bundled_package.parent}", flush=True)
        return
    try:
        if importlib.util.find_spec("google_ai") is not None:
            print("Using google_ai already available on kernel path", flush=True)
            return
    except Exception:
        pass
    deadline = time.monotonic() + REPO_BOOTSTRAP_WAIT_SECONDS
    last_snapshot: list[str] = []
    while True:
        repo_zip_path: Path | None = None
        repo_tree_root: Path | None = None
        flat_repo_root: Path | None = None
        snapshot: list[str] = []
        for path in INPUT_ROOT.rglob("*"):
            if path.is_file():
                snapshot.append(str(path.relative_to(INPUT_ROOT)))
                if len(snapshot) >= 40:
                    break
        last_snapshot = snapshot
        for path in INPUT_ROOT.rglob("repo_bundle.zip"):
            if path.is_file():
                repo_zip_path = path
                break
        if repo_zip_path is None:
            for init_path in INPUT_ROOT.rglob("__init__.py"):
                if init_path.parent.name == "google_ai":
                    repo_tree_root = init_path.parent.parent
                    break
                parent = init_path.parent
                if all((parent / name).is_file() for name in ("__init__.py", "client.py", "exceptions.py", "secrets.py")):
                    flat_repo_root = parent
                    break
        if repo_zip_path is not None:
            if WORK_REPO.exists():
                shutil.rmtree(WORK_REPO)
            WORK_REPO.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(repo_zip_path) as zip_file:
                zip_file.extractall(WORK_REPO)
            sys.path.insert(0, str(WORK_REPO))
            print(f"Bootstrapped repo bundle from {repo_zip_path}", flush=True)
            return
        if repo_tree_root is not None:
            sys.path.insert(0, str(repo_tree_root))
            print(f"Bootstrapped repo bundle from {repo_tree_root}", flush=True)
            return
        if flat_repo_root is not None:
            if WORK_REPO.exists():
                shutil.rmtree(WORK_REPO)
            package_root = WORK_REPO / "google_ai"
            package_root.mkdir(parents=True, exist_ok=True)
            for name in ("__init__.py", "client.py", "exceptions.py", "secrets.py"):
                shutil.copy2(flat_repo_root / name, package_root / name)
            sys.path.insert(0, str(WORK_REPO))
            print(f"Bootstrapped flat repo bundle from {flat_repo_root}", flush=True)
            return
        if time.monotonic() >= deadline:
            print(
                (
                    "Repo bundle not found under /kaggle/input; relying on kernel sources only "
                    f"visible={last_snapshot} wait_s={REPO_BOOTSTRAP_WAIT_SECONDS}"
                ),
                flush=True,
            )
            return
        time.sleep(5)
from cryptography.fernet import Fernet  # noqa: E402
from telethon import TelegramClient, functions  # noqa: E402
from telethon.sessions import StringSession  # noqa: E402

MODEL = "models/gemma-3-27b-it"
SCREEN_MODEL = MODEL
EXTRACT_MODEL = MODEL
GOOGLE_KEY_ENV = "GOOGLE_API_KEY2"
GOOGLE_FALLBACK_KEY_ENV = "GOOGLE_API_KEY"
GOOGLE_ACCOUNT_ENV = "GOOGLE_API_LOCALNAME2"
GOOGLE_ACCOUNT_FALLBACK_ENV = "GOOGLE_API_LOCALNAME"
LLM_TIMEOUT_SECONDS = 120
_GEMMA_CLIENTS: dict[str, Any] = {}
_SUPABASE_CLIENT: Any | None = None
_LLM_GATEWAY_LOGGED = False

URL_RE = re.compile(r"https?://[^\s<>()]+", re.I)
USERNAME_RE = re.compile(r"(?<!\w)@([A-Za-z0-9_]{4,64})")
PHONE_RE = re.compile(r"(?:(?:\+7|8)[\s(.-]*)?(?:\d[\s().-]*){10,11}")
DATE_RE = re.compile(
    r"\b(?:\d{1,2}[./]\d{1,2}(?:[./]\d{2,4})?|\d{1,2}\s+(?:январ|феврал|март|апрел|мая|май|июн|июл|август|сентябр|октябр|ноябр|декабр)[а-я]*)\b",
    re.I,
)
TIME_RE = re.compile(r"\b([01]?\d|2[0-3]):([0-5]\d)\b")


def _normalize_model_name(model: str) -> str:
    raw = (model or "").strip()
    if raw.startswith("models/"):
        return raw
    return f"models/{raw}"


def refresh_runtime_settings() -> None:
    global MODEL, SCREEN_MODEL, EXTRACT_MODEL, GOOGLE_KEY_ENV, GOOGLE_ACCOUNT_ENV, LLM_TIMEOUT_SECONDS
    global _GEMMA_CLIENTS, _SUPABASE_CLIENT, _LLM_GATEWAY_LOGGED
    MODEL = (os.getenv("GUIDE_MONITORING_MODEL") or os.getenv("GUIDE_MONITORING_SCREEN_MODEL") or "models/gemma-3-27b-it").strip()
    SCREEN_MODEL = (os.getenv("GUIDE_MONITORING_SCREEN_MODEL") or MODEL).strip()
    EXTRACT_MODEL = (os.getenv("GUIDE_MONITORING_EXTRACT_MODEL") or MODEL).strip()
    GOOGLE_KEY_ENV = (os.getenv("GUIDE_MONITORING_GOOGLE_KEY_ENV") or "GOOGLE_API_KEY2").strip() or "GOOGLE_API_KEY2"
    GOOGLE_ACCOUNT_ENV = (os.getenv("GUIDE_MONITORING_GOOGLE_ACCOUNT_ENV") or "GOOGLE_API_LOCALNAME2").strip() or "GOOGLE_API_LOCALNAME2"
    try:
        LLM_TIMEOUT_SECONDS = max(30, int(float((os.getenv("GUIDE_MONITORING_LLM_TIMEOUT_SEC") or "120").strip() or 120)))
    except Exception:
        LLM_TIMEOUT_SECONDS = 120
    _GEMMA_CLIENTS = {}
    _SUPABASE_CLIENT = None
    _LLM_GATEWAY_LOGGED = False


class _GuideSecretsProviderAdapter:
    def __init__(self, base: Any):
        self.base = base

    def get_secret(self, name: str) -> str | None:
        if name == "GOOGLE_API_KEY":
            return self.base.get_secret(GOOGLE_KEY_ENV) or self.base.get_secret(GOOGLE_FALLBACK_KEY_ENV)
        return self.base.get_secret(name)

    def get_secret_pool(self, prefix: str) -> list[str]:
        if prefix == "GOOGLE_API_KEY":
            keys: list[str] = []
            primary = self.get_secret("GOOGLE_API_KEY")
            if primary:
                keys.append(primary)
            return keys
        getter = getattr(self.base, "get_secret_pool", None)
        if callable(getter):
            return list(getter(prefix) or [])
        return []


def _build_supabase_client() -> Any | None:
    if (os.getenv("SUPABASE_DISABLED") or "").strip() == "1":
        return None
    base_url = (os.getenv("SUPABASE_URL") or "").strip().rstrip("/")
    key = (os.getenv("SUPABASE_SERVICE_KEY") or os.getenv("SUPABASE_KEY") or "").strip()
    if not base_url or not key:
        return None
    from supabase import create_client
    from supabase.client import ClientOptions

    options = ClientOptions()
    options.schema = (os.getenv("SUPABASE_SCHEMA") or "public").strip() or "public"
    return create_client(base_url, key, options=options)


def _get_supabase_client() -> Any | None:
    global _SUPABASE_CLIENT
    if _SUPABASE_CLIENT is None:
        _SUPABASE_CLIENT = _build_supabase_client()
    return _SUPABASE_CLIENT


def _guide_account_name() -> str | None:
    return (os.getenv(GOOGLE_ACCOUNT_ENV) or os.getenv(GOOGLE_ACCOUNT_FALLBACK_ENV) or "").strip() or None


def _log_llm_gateway_once() -> None:
    global _LLM_GATEWAY_LOGGED
    if _LLM_GATEWAY_LOGGED:
        return
    print(
        (
            "Guide monitor llm_gateway="
            f"google_ai key_env={GOOGLE_KEY_ENV} "
            f"account_env={GOOGLE_ACCOUNT_ENV} "
            f"account_name={_guide_account_name() or '-'} "
            f"supabase={'yes' if _get_supabase_client() is not None else 'no'} "
            f"timeout={LLM_TIMEOUT_SECONDS}s "
            f"reserve_fallback={os.getenv('GOOGLE_AI_ALLOW_RESERVE_FALLBACK', '1')} "
            f"local_fallback={os.getenv('GOOGLE_AI_LOCAL_LIMITER_FALLBACK', '1')}"
        ),
        flush=True,
    )
    _LLM_GATEWAY_LOGGED = True


def _get_gemma_client(consumer: str) -> Any:
    client = _GEMMA_CLIENTS.get(consumer)
    if client is None:
        _bootstrap_repo_bundle()
        from google_ai import GoogleAIClient, SecretsProvider

        _log_llm_gateway_once()
        client = GoogleAIClient(
            supabase_client=_get_supabase_client(),
            secrets_provider=_GuideSecretsProviderAdapter(SecretsProvider()),
            consumer=consumer,
            account_name=_guide_account_name(),
        )
        _GEMMA_CLIENTS[consumer] = client
        print(
            (
                f"[gemma:client] consumer={consumer} "
                f"model_family=gemma-only account={_guide_account_name() or '-'} "
                f"key_env={GOOGLE_KEY_ENV}"
            ),
            flush=True,
        )
    return client


def _find_file(name: str) -> Path:
    for path in INPUT_ROOT.rglob(name):
        if path.is_file():
            return path
    raise RuntimeError(f"{name} not found under {INPUT_ROOT}")


def load_runtime_config() -> dict[str, Any]:
    config_path = _find_file("config.json")
    secrets_path = _find_file("secrets.enc")
    key_path = _find_file("fernet.key")
    config = json.loads(config_path.read_text(encoding="utf-8"))
    decrypted = Fernet(key_path.read_bytes()).decrypt(secrets_path.read_bytes())
    secrets = json.loads(decrypted.decode("utf-8"))
    if not isinstance(config, dict) or not isinstance(secrets, dict):
        raise RuntimeError("Invalid config/secrets payload")
    for key, value in secrets.items():
        if value is None:
            continue
        os.environ.setdefault(str(key), str(value))
    return config


def _resolve_auth_bundle() -> tuple[str | None, dict[str, Any] | None]:
    bundle_env = (os.getenv("GUIDE_MONITORING_AUTH_BUNDLE_ENV") or "").strip()
    candidates = [bundle_env, "TELEGRAM_AUTH_BUNDLE_S22", "TELEGRAM_AUTH_BUNDLE_E2E"]
    for key in candidates:
        if not key:
            continue
        raw = (os.getenv(key) or "").strip()
        if not raw:
            continue
        decoded = base64.urlsafe_b64decode(raw.encode("ascii")).decode("utf-8")
        payload = json.loads(decoded)
        if isinstance(payload, dict) and str(payload.get("session") or "").strip():
            return key, payload
    return None, None


async def create_client() -> TelegramClient:
    api_id = int((os.getenv("TG_API_ID") or os.getenv("TELEGRAM_API_ID") or "0").strip() or 0)
    api_hash = (os.getenv("TG_API_HASH") or os.getenv("TELEGRAM_API_HASH") or "").strip()
    if not api_id or not api_hash:
        raise RuntimeError("Missing TG_API_ID/TG_API_HASH")
    _bundle_env, bundle = _resolve_auth_bundle()
    session = str((bundle or {}).get("session") or os.getenv("TG_SESSION") or os.getenv("TELEGRAM_SESSION") or "").strip()
    if not session:
        raise RuntimeError("Missing TELEGRAM auth bundle or session")
    client = TelegramClient(
        StringSession(session),
        api_id,
        api_hash,
        device_model=str((bundle or {}).get("device_model") or "Kaggle Guide Monitor"),
        system_version=str((bundle or {}).get("system_version") or "Linux"),
        app_version=str((bundle or {}).get("app_version") or "1.0"),
        lang_code=str((bundle or {}).get("lang_code") or "ru"),
        system_lang_code=str((bundle or {}).get("system_lang_code") or "ru"),
    )
    await client.connect()
    if not await client.is_user_authorized():
        raise RuntimeError("Telethon client is not authorized")
    return client


def _message_text(message: Any) -> str:
    return str(getattr(message, "message", None) or getattr(message, "text", None) or "").strip()


def _message_media_kind(message: Any) -> str | None:
    if getattr(message, "photo", None):
        return "photo"
    if getattr(message, "video", None):
        return "video"
    doc = getattr(message, "document", None)
    if doc is None:
        return None
    mime = str(getattr(doc, "mime_type", None) or "").lower()
    if mime.startswith("video/"):
        return "video"
    if mime.startswith("image/"):
        return "photo"
    return None


def _reactions_payload(message: Any) -> tuple[int | None, dict[str, int] | None]:
    reactions = getattr(message, "reactions", None)
    if not reactions or not getattr(reactions, "results", None):
        return None, None
    out: dict[str, int] = {}
    total = 0
    for item in getattr(reactions, "results", []) or []:
        count = int(getattr(item, "count", 0) or 0)
        reaction = getattr(item, "reaction", None)
        emoji = str(getattr(reaction, "emoticon", None) or getattr(reaction, "document_id", None) or reaction or "")
        if not emoji:
            emoji = "reaction"
        out[emoji] = count
        total += count
    return total or None, out or None


@dataclass(slots=True)
class ScannedPost:
    message_id: int
    grouped_id: int | None
    post_date: datetime
    source_url: str
    text: str
    views: int | None
    forwards: int | None
    reactions_total: int | None
    reactions_json: dict[str, int] | None
    media_refs: list[dict[str, Any]]


def collapse_ws(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_title_key(value: str | None) -> str:
    text = collapse_ws(value).lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[^a-zа-яё0-9]+", " ", text, flags=re.I)
    text = re.sub(r"\b(?:экскурсия|экскурсии|прогулка|прогулки|тур|маршрут|авторская|пешеходная|поездка|путешествие)\b", " ", text, flags=re.I)
    return collapse_ws(text)


def build_source_fingerprint(*, title_normalized: str, date_iso: str | None, time_text: str | None) -> str:
    import hashlib

    payload = "|".join([str(title_normalized or ""), str(date_iso or ""), str(time_text or "")])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _collapse_group(messages: list[Any], *, username: str) -> ScannedPost | None:
    ordered = sorted(messages, key=lambda item: int(getattr(item, "id", 0) or 0))
    anchor = next((msg for msg in ordered if _message_text(msg)), None) or ordered[-1]
    anchor_id = int(getattr(anchor, "id", 0) or 0)
    post_date = getattr(anchor, "date", None) or getattr(ordered[-1], "date", None) or datetime.now(timezone.utc)
    if post_date.tzinfo is None:
        post_date = post_date.replace(tzinfo=timezone.utc)
    text = "\n".join(_message_text(msg) for msg in ordered if _message_text(msg)).strip()
    views = max((getattr(msg, "views", None) or 0) for msg in ordered) or None
    forwards = max((getattr(msg, "forwards", None) or 0) for msg in ordered) or None
    reactions_total = None
    reactions_json = None
    for msg in ordered:
        total, payload = _reactions_payload(msg)
        if total is not None and ((reactions_total or -1) < total):
            reactions_total = total
            reactions_json = payload
    media_refs: list[dict[str, Any]] = []
    for msg in ordered:
        kind = _message_media_kind(msg)
        if not kind:
            continue
        media_refs.append(
            {
                "message_id": int(getattr(msg, "id", 0) or 0),
                "kind": kind,
                "grouped_id": int(getattr(msg, "grouped_id", 0) or 0) or None,
            }
        )
    if not text and not media_refs:
        return None
    return ScannedPost(
        message_id=anchor_id,
        grouped_id=int(getattr(anchor, "grouped_id", 0) or 0) or None,
        post_date=post_date.astimezone(timezone.utc),
        source_url=f"https://t.me/{username}/{anchor_id}",
        text=text,
        views=views,
        forwards=forwards,
        reactions_total=reactions_total,
        reactions_json=reactions_json,
        media_refs=media_refs,
    )


async def scan_source_posts(client: TelegramClient, *, username: str, limit: int, days_back: int) -> tuple[dict[str, Any], list[ScannedPost]]:
    entity = await client.get_entity(username)
    source_title = str(getattr(entity, "title", None) or getattr(entity, "first_name", None) or "").strip() or None
    about_text = ""
    about_links: list[str] = []
    try:
        full = await client(functions.channels.GetFullChannelRequest(channel=entity))
        about_text = str(getattr(full.full_chat, "about", None) or "").strip()
    except Exception:
        try:
            full = await client(functions.users.GetFullUserRequest(id=entity))
            about_text = str(getattr(full.full_user, "about", None) or "").strip()
        except Exception:
            about_text = ""
    if about_text:
        seen: set[str] = set()
        for token in about_text.replace("\n", " ").split():
            raw = str(token or "").strip("()[]{}<>.,!?:;\"'")
            if raw.startswith("http://") or raw.startswith("https://"):
                if raw not in seen:
                    seen.add(raw)
                    about_links.append(raw)
    messages = await client.get_messages(entity, limit=max(1, int(limit)))
    cutoff = datetime.now(timezone.utc) - timedelta(days=max(1, int(days_back)))
    singles: list[Any] = []
    grouped: dict[int, list[Any]] = {}
    for msg in messages:
        msg_date = getattr(msg, "date", None)
        if msg_date is None:
            continue
        if msg_date.tzinfo is None:
            msg_date = msg_date.replace(tzinfo=timezone.utc)
        if msg_date.astimezone(timezone.utc) < cutoff:
            continue
        if getattr(msg, "action", None):
            continue
        grouped_id = int(getattr(msg, "grouped_id", 0) or 0)
        if grouped_id:
            grouped.setdefault(grouped_id, []).append(msg)
        else:
            singles.append(msg)
    posts: list[ScannedPost] = []
    for msg in singles:
        collapsed = _collapse_group([msg], username=username)
        if collapsed:
            posts.append(collapsed)
    for group in grouped.values():
        collapsed = _collapse_group(group, username=username)
        if collapsed:
            posts.append(collapsed)
    posts.sort(key=lambda item: (item.post_date, item.message_id), reverse=True)
    return {"source_title": source_title, "about_text": about_text or None, "about_links": about_links}, posts


def prefilter_flags(post: ScannedPost) -> dict[str, Any]:
    text = collapse_ws(post.text).lower()
    return {
        "has_date_signal": bool(DATE_RE.search(text) or "завтра" in text or "сегодня" in text),
        "has_time_signal": bool(TIME_RE.search(text)),
        "has_price_signal": any(token in text for token in ("стоимость", "цена", "руб", "₽")),
        "has_booking_signal": bool(URL_RE.search(text) or USERNAME_RE.search(text) or PHONE_RE.search(text) or "запись" in text or "бронир" in text),
        "has_status_signal": any(token in text for token in ("мест нет", "лист ожидания", "sold out", "перенос", "отмена", "осталось", "последние места")),
        "has_group_signal": any(token in text for token in ("по запросу", "организованные группы", "для групп", "школьн", "семь")),
        "has_excursion_keywords": any(token in text for token in ("экскурс", "прогул", "маршрут", "путешеств", "тур ")),
        "grouped_album_present": bool(post.grouped_id),
        "message_url": post.source_url,
    }


def prefilter_pass(post: ScannedPost, source_kind: str) -> bool:
    flags = prefilter_flags(post)
    if not flags["has_excursion_keywords"]:
        return False
    if source_kind == "aggregator" and not any(token in collapse_ws(post.text).lower() for token in ("авторская", "приглашаем", "пешеходная")):
        return False
    return any(flags[key] for key in ("has_date_signal", "has_booking_signal", "has_status_signal", "grouped_album_present"))


def split_text_chunks(text: str, *, limit: int = 8) -> list[dict[str, str]]:
    chunks: list[dict[str, str]] = []
    for idx, block in enumerate(re.split(r"\n{2,}", str(text or "").strip()), start=1):
        cleaned = collapse_ws(block)
        if not cleaned:
            continue
        chunks.append({"id": f"T{idx}", "text": cleaned[:700]})
        if len(chunks) >= limit:
            break
    if not chunks and collapse_ws(text):
        chunks.append({"id": "T1", "text": collapse_ws(text)[:700]})
    return chunks


def _extract_json(raw: str) -> dict[str, Any] | None:
    cleaned = str(raw or "").strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```[a-zA-Z0-9_-]*\n", "", cleaned)
        cleaned = cleaned.replace("```", "")
    try:
        data = json.loads(cleaned)
        return data if isinstance(data, dict) else None
    except Exception:
        pass
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start >= 0 and end > start:
        try:
            data = json.loads(cleaned[start : end + 1])
            return data if isinstance(data, dict) else None
        except Exception:
            return None
    return None


async def ask_gemma(
    model: str,
    prompt: str,
    *,
    consumer: str,
    max_output_tokens: int = 2200,
) -> dict[str, Any] | None:
    if not (os.getenv(GOOGLE_KEY_ENV) or os.getenv(GOOGLE_FALLBACK_KEY_ENV) or "").strip():
        raise RuntimeError(f"{GOOGLE_KEY_ENV} is missing in Kaggle runtime")
    client = _get_gemma_client(consumer)
    raw, _usage = await asyncio.wait_for(
        client.generate_content_async(
            model=model,
            prompt=prompt,
            generation_config={"temperature": 0},
            max_output_tokens=max_output_tokens,
        ),
        timeout=LLM_TIMEOUT_SECONDS,
    )
    return _extract_json(raw or "")


async def screen_post(source_payload: dict[str, Any], post: ScannedPost, flags: dict[str, Any]) -> dict[str, Any]:
    prompt = (
        "You classify one Telegram post from a guide-related excursions source.\n"
        "Return only JSON with keys: decision, post_kind, extract_mode, digest_eligible_default, contains_future_public_signal, contains_past_report_signal, reasons, confidence.\n"
        "Allowed decision: ignore, announce, status_update, template_only.\n"
        "Allowed post_kind: announce_single, announce_multi, status_update, reportage, template_signal, on_demand_offer, mixed_or_non_target.\n"
        "Allowed extract_mode: none, announce, status, template.\n"
        "Do not invent missing facts.\n"
        f"Input:\n{json.dumps({'source': source_payload, 'post': {'text_chunks': split_text_chunks(post.text), 'prefilter_flags': flags, 'message_id': post.message_id, 'post_date_utc': post.post_date.isoformat(), 'message_url': post.source_url}}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        SCREEN_MODEL,
        prompt,
        consumer="guide_scout_screen",
        max_output_tokens=600,
    )
    if not isinstance(data, dict):
        return {
            "decision": "ignore",
            "post_kind": "mixed_or_non_target",
            "extract_mode": "none",
            "digest_eligible_default": "mixed",
            "contains_future_public_signal": False,
            "contains_past_report_signal": False,
            "reasons": ["llm_parse_failed"],
            "confidence": "low",
        }
    return {
        "decision": str(data.get("decision") or "ignore"),
        "post_kind": str(data.get("post_kind") or "mixed_or_non_target"),
        "extract_mode": str(data.get("extract_mode") or "none"),
        "digest_eligible_default": str(data.get("digest_eligible_default") or "mixed"),
        "contains_future_public_signal": bool(data.get("contains_future_public_signal")),
        "contains_past_report_signal": bool(data.get("contains_past_report_signal")),
        "reasons": data.get("reasons") if isinstance(data.get("reasons"), list) else [],
        "confidence": str(data.get("confidence") or "low"),
    }


def _clean_occurrence_payload(item: dict[str, Any], *, post: ScannedPost) -> dict[str, Any] | None:
    title = collapse_ws(item.get("canonical_title"))
    if not title:
        return None
    title_normalized = collapse_ws(item.get("title_normalized")) or normalize_title_key(title)
    if not title_normalized:
        return None
    date_iso = collapse_ws(item.get("date") or item.get("date_iso")) or None
    time_text = collapse_ws(item.get("time") or item.get("time_text")) or None
    out = {
        "canonical_title": title,
        "title_normalized": title_normalized,
        "date": date_iso,
        "time": time_text,
        "city": collapse_ws(item.get("city")) or None,
        "meeting_point": collapse_ws(item.get("meeting_point")) or None,
        "audience_fit": [collapse_ws(x) for x in (item.get("audience_fit") or []) if collapse_ws(x)],
        "price_text": collapse_ws(item.get("price_text")) or None,
        "booking_text": collapse_ws(item.get("booking_text")) or None,
        "booking_url": collapse_ws(item.get("booking_url")) or None,
        "channel_url": collapse_ws(item.get("channel_url")) or post.source_url,
        "status": collapse_ws(item.get("status")) or "scheduled",
        "seats_text": collapse_ws(item.get("seats_text")) or None,
        "summary_one_liner": collapse_ws(item.get("summary_one_liner")) or None,
        "digest_blurb": collapse_ws(item.get("digest_blurb")) or collapse_ws(item.get("summary_one_liner")) or None,
        "digest_eligible": bool(item.get("digest_eligible")),
        "digest_eligibility_reason": collapse_ws(item.get("digest_eligibility_reason")) or None,
        "is_last_call": bool(item.get("is_last_call")),
        "post_kind": collapse_ws(item.get("post_kind")) or "announce_single",
        "availability_mode": collapse_ws(item.get("availability_mode")) or "scheduled_public",
        "guide_names": [collapse_ws(x) for x in (item.get("guide_names") or []) if collapse_ws(x)],
        "organizer_names": [collapse_ws(x) for x in (item.get("organizer_names") or []) if collapse_ws(x)],
        "source_fingerprint": collapse_ws(item.get("source_fingerprint")) or build_source_fingerprint(title_normalized=title_normalized, date_iso=date_iso, time_text=time_text),
        "fact_pack": item.get("fact_pack") if isinstance(item.get("fact_pack"), dict) else {},
        "fact_claims": item.get("fact_claims") if isinstance(item.get("fact_claims"), list) else [],
        "template_hint": item.get("template_hint") if isinstance(item.get("template_hint"), dict) else {},
        "profile_hint": item.get("profile_hint") if isinstance(item.get("profile_hint"), dict) else {},
    }
    return out


async def extract_post(source_payload: dict[str, Any], post: ScannedPost, flags: dict[str, Any], screen: dict[str, Any]) -> list[dict[str, Any]]:
    prompt = (
        "You extract guide-excursion facts from one Telegram post.\n"
        "Return only JSON with key occurrences.\n"
        "Each occurrence must be an object with keys: canonical_title, title_normalized, date, time, city, meeting_point, audience_fit, price_text, booking_text, booking_url, channel_url, status, seats_text, summary_one_liner, digest_blurb, digest_eligible, digest_eligibility_reason, is_last_call, post_kind, availability_mode, guide_names, organizer_names, fact_claims, template_hint, profile_hint, fact_pack.\n"
        "Rules:\n"
        "- Extract only guide excursions or direct updates about a specific excursion occurrence.\n"
        "- Ignore past occurrences for MVP.\n"
        "- booking_url can be http(s), t.me link, or tel:.\n"
        "- digest_blurb must be short and clean.\n"
        "- Use only facts visible in the source. Do not invent.\n"
        "- fact_claims must use claim_role from: anchor, support, status_delta, template_hint, guide_profile_hint.\n"
        f"Input:\n{json.dumps({'source': source_payload, 'screen': screen, 'post': {'message_id': post.message_id, 'post_date_utc': post.post_date.isoformat(), 'message_url': post.source_url, 'text_chunks': split_text_chunks(post.text), 'prefilter_flags': flags}}, ensure_ascii=False)}"
    )
    data = await ask_gemma(
        EXTRACT_MODEL,
        prompt,
        consumer="guide_scout_tier1_extract",
        max_output_tokens=2400,
    )
    items = data.get("occurrences") if isinstance(data, dict) else []
    cleaned: list[dict[str, Any]] = []
    if isinstance(items, list):
        for item in items:
            if not isinstance(item, dict):
                continue
            occurrence = _clean_occurrence_payload(item, post=post)
            if occurrence:
                cleaned.append(occurrence)
    return cleaned


def _llm_status_from_exception(exc: Exception) -> str:
    rate_limit_error = None
    provider_error = None
    try:
        from google_ai.exceptions import ProviderError, RateLimitError

        provider_error = ProviderError
        rate_limit_error = RateLimitError
    except Exception:
        provider_error = None
        rate_limit_error = None
    if rate_limit_error is not None and isinstance(exc, rate_limit_error):
        blocked = (getattr(exc, "blocked_reason", None) or "unknown").strip() or "unknown"
        return f"llm_deferred_rate_limit:{blocked}"
    if provider_error is not None and isinstance(exc, provider_error) and int(getattr(exc, "status_code", 0) or 0) == 429:
        return "llm_deferred_provider_429"
    if isinstance(exc, asyncio.TimeoutError):
        return "llm_deferred_timeout"
    return f"llm_error:{type(exc).__name__}"


def _summarize_source_stats(posts: list[dict[str, Any]]) -> dict[str, int]:
    llm_ok = 0
    llm_deferred = 0
    llm_error = 0
    for post in posts:
        status = str(post.get("llm_status") or "").strip()
        if status == "ok":
            llm_ok += 1
        elif status.startswith("llm_deferred"):
            llm_deferred += 1
        elif status not in {"", "skipped_prefilter"}:
            llm_error += 1
    return {
        "posts_total": len(posts),
        "prefilter_true": sum(1 for post in posts if bool(post.get("prefilter_passed"))),
        "llm_ok": llm_ok,
        "llm_deferred": llm_deferred,
        "llm_error": llm_error,
        "occurrences_total": sum(len(post.get("occurrences") or []) for post in posts),
    }


def _summarize_run_stats(sources_output: list[dict[str, Any]]) -> dict[str, int]:
    totals = {
        "sources_total": len(sources_output),
        "posts_total": 0,
        "prefilter_true": 0,
        "llm_ok": 0,
        "llm_deferred": 0,
        "llm_error": 0,
        "occurrences_total": 0,
    }
    for source_payload in sources_output:
        stats = source_payload.get("stats") if isinstance(source_payload.get("stats"), dict) else {}
        for key in ("posts_total", "prefilter_true", "llm_ok", "llm_deferred", "llm_error", "occurrences_total"):
            totals[key] += int(stats.get(key) or 0)
    return totals


async def process_source(client: TelegramClient, source_payload: dict[str, Any], *, limit: int, days_back: int) -> dict[str, Any]:
    username = str(source_payload.get("username") or "").strip()
    source_kind = str(source_payload.get("source_kind") or "").strip()
    print(f"[source:start] @{username} kind={source_kind or 'unknown'} limit={limit} days_back={days_back}", flush=True)
    meta, posts = await scan_source_posts(client, username=username, limit=limit, days_back=days_back)
    out_posts: list[dict[str, Any]] = []
    errors: list[str] = []
    for post in posts:
        flags = prefilter_flags(post)
        passes = prefilter_pass(post, source_kind)
        payload: dict[str, Any] = {
            "message_id": post.message_id,
            "grouped_id": post.grouped_id,
            "post_date": post.post_date.isoformat(),
            "source_url": post.source_url,
            "text": post.text,
            "views": post.views,
            "forwards": post.forwards,
            "reactions_total": post.reactions_total,
            "reactions_json": post.reactions_json,
            "media_refs": post.media_refs,
            "prefilter_passed": passes,
            "prefilter_flags": flags,
            "llm_status": "skipped_prefilter",
            "screen": {
                "decision": "ignore",
                "post_kind": "mixed_or_non_target",
                "extract_mode": "none",
                "digest_eligible_default": "mixed",
                "contains_future_public_signal": False,
                "contains_past_report_signal": False,
                "reasons": ["prefilter_false"],
                "confidence": "low",
            },
            "occurrences": [],
        }
        if passes:
            try:
                screen = await screen_post(
                    {
                        "username": username,
                        "source_kind": source_kind,
                        "title": meta.get("source_title"),
                        "display_name": source_payload.get("display_name"),
                        "marketing_name": source_payload.get("marketing_name"),
                    },
                    post,
                    flags,
                )
                payload["screen"] = screen
                if screen.get("extract_mode") != "none" and screen.get("decision") != "ignore":
                    payload["occurrences"] = await extract_post(
                        {
                            "username": username,
                            "source_kind": source_kind,
                            "title": meta.get("source_title"),
                            "display_name": source_payload.get("display_name"),
                            "marketing_name": source_payload.get("marketing_name"),
                        },
                        post,
                        flags,
                        screen,
                    )
                payload["llm_status"] = "ok"
            except Exception as exc:
                error_kind = _llm_status_from_exception(exc)
                payload["llm_status"] = error_kind
                payload["screen"] = {
                    "decision": "ignore",
                    "post_kind": "mixed_or_non_target",
                    "extract_mode": "none",
                    "digest_eligible_default": "mixed",
                    "contains_future_public_signal": False,
                    "contains_past_report_signal": False,
                    "reasons": [error_kind],
                    "confidence": "low",
                }
                errors.append(f"message_id={post.message_id}: {error_kind}: {exc}")
        out_posts.append(payload)
    source_status = "partial" if errors else "ok"
    print(
        (
            f"[source:done] @{username} status={source_status} "
            f"posts={len(posts)} extracted={sum(len((item.get('occurrences') or [])) for item in out_posts)} "
            f"errors={len(errors)}"
        ),
        flush=True,
    )
    stats = _summarize_source_stats(out_posts)
    return {
        "username": username,
        "source_title": meta.get("source_title"),
        "source_kind": source_kind,
        "about_text": meta.get("about_text"),
        "about_links": meta.get("about_links"),
        "source_status": source_status,
        "posts_scanned": len(posts),
        "stats": stats,
        "errors": errors,
        "posts": out_posts,
    }


async def main() -> None:
    config = load_runtime_config()
    _bootstrap_repo_bundle()
    refresh_runtime_settings()
    run_id = str(config.get("run_id") or f"guide_kaggle_{int(datetime.now(timezone.utc).timestamp())}")
    started_at = datetime.now(timezone.utc).isoformat()
    sources = [item for item in (config.get("sources") or []) if isinstance(item, dict)]
    print(
        (
            f"Guide monitor run_id={run_id} mode={config.get('scan_mode') or 'full'} "
            f"sources={len(sources)} limit={int(config.get('limit_per_source') or 25)} "
            f"days_back={int(config.get('days_back') or 7)} "
            f"screen_model={SCREEN_MODEL} extract_model={EXTRACT_MODEL}"
        ),
        flush=True,
    )
    client = await create_client()
    partial = False
    sources_output: list[dict[str, Any]] = []
    try:
        limit = int(config.get("limit_per_source") or 25)
        days_back = int(config.get("days_back") or 7)
        for source in sources:
            if not isinstance(source, dict):
                continue
            try:
                result = await process_source(client, source, limit=limit, days_back=days_back)
            except Exception as exc:
                partial = True
                result = {
                    "username": str(source.get("username") or ""),
                    "source_title": str(source.get("title") or "") or None,
                    "source_kind": str(source.get("source_kind") or "") or None,
                    "source_status": "error",
                    "posts_scanned": 0,
                    "errors": [f"{type(exc).__name__}: {exc}"],
                    "posts": [],
                }
            if result.get("source_status") != "ok":
                partial = True
            sources_output.append(result)
    finally:
        await client.disconnect()
    stats = _summarize_run_stats(sources_output)
    payload = {
        "schema_version": 1,
        "run_id": run_id,
        "scan_mode": str(config.get("scan_mode") or "full"),
        "started_at": started_at,
        "finished_at": datetime.now(timezone.utc).isoformat(),
        "partial": partial,
        "stats": stats,
        "sources": sources_output,
    }
    RESULT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(
        (
            f"Guide monitor completed partial={partial} "
            f"sources={len(sources_output)} "
            f"posts={sum(int(item.get('posts_scanned') or 0) for item in sources_output)}"
        ),
        flush=True,
    )
    print(
        (
            "Guide monitor stats "
            f"posts_total={stats['posts_total']} "
            f"prefilter_true={stats['prefilter_true']} "
            f"llm_ok={stats['llm_ok']} "
            f"llm_deferred={stats['llm_deferred']} "
            f"llm_error={stats['llm_error']} "
            f"occurrences_total={stats['occurrences_total']}"
        ),
        flush=True,
    )
    print(f"Wrote {RESULT_PATH}")




In [ ]:
print('Guide notebook bootstrap complete', flush=True)
print(f'Guide notebook runner file={__file__}', flush=True)
print(f'Guide notebook embedded google_ai root={_GUIDE_EMBEDDED_ROOT}', flush=True)


In [ ]:
import asyncio
import nest_asyncio

def _guide_run_main_sync() -> None:
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    if loop.is_closed():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    if loop.is_running():
        nest_asyncio.apply(loop)
    loop.run_until_complete(main())

_guide_run_main_sync()
